# Домашнее задание: Сбор данных и разметка: от формулировки задачи до крауда

В этом кейсе вы пройдёте путь **от постановки бизнеса** до **пайплайна гибридной разметки**:  
Постановка задачи → разметка zero shot промптом с помощью LLM → оценка качества → Улучшение качества промпта: few-shot, cot и другие способы → Оценка уверенности ответа




### Установка зависимостей

In [2]:
%pip install "torch>=2.6,<3" "transformers==4.57.1" "datasets>=4,<5" accelerate scikit-learn pandas tqdm matplotlib


  Using cached transformers-4.57.1-py3-none-any.whl.metadata (43 kB)
Using cached transformers-4.57.1-py3-none-any.whl (12.0 MB)
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import torch, json, random, re, pandas as pd, numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sklearn.metrics import precision_recall_fscore_support
from tqdm.auto import tqdm
torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print('Device:', device)


W0918 10:20:58.052000 39216 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Device: cuda


## 1. Постановка задачи


**Контекст (от лица бизнеса):**
Наша компания разрабатывает финтех-приложение с поддержкой пользователей через чат-бот.
Мы хотим автоматически определять тему запроса клиента (например: "блокировка карты", "потеря ПИН-кода", "перевыпуск карты", и т.д.), чтобы быстро направлять клиента к нужному решению. Требуется получить данные для задачи




**Описание задачи:**
> "Для каждого входящего текстового сообщения пользователя автоматически определить одну из тематик (например, balance, card_not_working, transfer, etc.)"

## 2. Требования и бизнес-метрики – 1 балл

Предложите не менее 2 бизнес-метрик, которые может хотеть оптимизировать бизнес относительно процесса разметки данных для данной задачи.


In [2]:
# ваш ответ тут

# ---- Ваш код здесь ----
print("""
Хорошие метрики определить невозможно из-за непонятного мне задания. Предположим так:

Бизнес-метрики для задачи получения данных:
- стоимость одной размеченной записи
- время получения полного набора данных

""")
# ---- Конец кода ----


Хорошие метрики определить невозможно из-за непонятного мне задания. Предположим так:

Бизнес-метрики для задачи получения данных:
- стоимость одной размеченной записи
- время получения полного набора данных




## 3. Сведение к ML-задаче – 2 балла



Сведите бизнес-задачу к задаче машинного обучения, опишите входные данные и метки:

- **Тип задачи**:
- **Объект**:
- **Метки**:

In [3]:
# ваш ответ тут

# ---- Ваш код здесь ----
print("""
- **Тип задачи**: многоклассовая классификация текста (multi-class text classification, single-label).
- **Объект**: текстовое сообщение пользователя. Размер фиксирован (например, берем первые 2048 символов).
- **Метки**: заранее определенные тематики службы поддержки (например, balance, card_not_working, transfer, etc.). 
 На первом этапе количесво классов фиксировано, но может быть расширено в будущем.
""")
# ---- Конец кода ----


- **Тип задачи**: многоклассовая классификация текста (multi-class text classification, single-label).
- **Объект**: текстовое сообщение пользователя. Размер фиксирован (например, берем первые 2048 символов).
- **Метки**: заранее определенные тематики службы поддержки (например, balance, card_not_working, transfer, etc.). 
 На первом этапе количесво классов фиксировано, но может быть расширено в будущем.



## 4. ML-метрики – 2 балла


Сформулируйте, какие метрики вы будете отслеживать в процессе сбора данных и получения разметки: как при помощи LLM, так и при помощи разметчиков в крауде

In [4]:
# ---- Ваш код здесь ----
print("""
- LLM разметчик: accuracy, ROC-AUC confidence. Ошибки формата оказались тоже важными.
- Разметчики в крауде: accuracy на контрольных заданиях (honeypots)

""")
# ---- Конец кода ----




- LLM разметчик: accuracy, ROC-AUC confidence. Ошибки формата оказались тоже важными.
- Разметчики в крауде: accuracy на контрольных заданиях (honeypots)




## 5. Данные и бейзлайн разметка

### 5.1 Загрузка и первичный анализ датасета

Посмотрим данные: примеры из датасета (попробуем разметить сами хотя бы 10 примеров), все типы меток, размеры выборок, распределение

In [5]:
# CSV из репозитория авторов: загрузчик не требует устаревшего dataset script.
# Источник и лицензия CC BY 4.0: https://github.com/PolyAI-LDN/task-specific-datasets/tree/master/banking_data
from datasets import load_dataset, ClassLabel
import pandas as pd

DATA_REVISION = "57ec275d8078af65b7731c2a98be812d844a6d6b"
DATA_URL = f"https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/{DATA_REVISION}/banking_data"
ds = load_dataset("csv", data_files={split: f"{DATA_URL}/{split}.csv" for split in ("train", "test")})
ds = ds.rename_column("category", "label")
label_names = sorted(set(ds["train"]["label"]))
assert len(label_names) == 77
assert set(ds["test"]["label"]) == set(label_names)
ds = ds.cast_column("label", ClassLabel(names=label_names))
# Сохраняем ds["train"], ds["test"], text и числовой label, как у исходного загрузчика.
# Соответствие ID -> метка задаёт ds["train"].features["label"].names; не берите ID из других версий.


# ---- Ваш код здесь ----
print("""
Считываем данные
""")
train_df = ds["train"].to_pandas()
test_df = ds["test"].to_pandas()
id2label = ds["train"].features["label"].names
train_df["label_name"] = train_df["label"].map(lambda i: id2label[i])
test_df["label_name"] = test_df["label"].map(lambda i: id2label[i])

print(ds["train"][0])
print(train_df.iloc[0])

# 1. Размеры выборок
print("Размеры выборок:")
print(f"  train: {len(train_df)} примеров")
print(f"  test:  {len(test_df)} примеров")
print(f"  классов: {len(id2label)}")

# 2. Все типы меток
print()
print("Все метки (id -> name):")
for i, name in enumerate(id2label):
    print(f"  {i:2d}  {name}")

# 3. Распределение меток
train_counts = train_df["label_name"].value_counts()
test_counts = test_df["label_name"].value_counts()
dist = pd.DataFrame({"train": train_counts, "test": test_counts}).fillna(0).astype(int)
dist["train_%"] = (100 * dist["train"] / dist["train"].sum()).round(2)
print()
print("Распределение меток (train): "
      f"min={train_counts.min()}, max={train_counts.max()}, "
      f"median={int(train_counts.median())}, mean={train_counts.mean():.1f}")
print("Самые частые классы:")
print(dist.sort_values("train", ascending=False).head(5).to_string())
print("Самые редкие классы:")
print(dist.sort_values("train").head(5).to_string())

# 4. Длина текстов (хватит ли для обучения?)
train_len = train_df["text"].str.split().str.len()
print()
print("Длина текста в словах (train): "
      f"min={train_len.min()}, median={int(train_len.median())}, "
      f"p95={int(train_len.quantile(0.95))}, max={train_len.max()}")

# 5. Примеры для ручной разметки: 10 случайных текстов из train.
# Сначала показываем только текст, чтобы попробовать разметить самим,
# затем раскрываем истинную метку и сравниваем.
sample = train_df.sample(n=4, random_state=42).reset_index(drop=True)
print()
print("Пробуем разметить сами (4 текста, слишком долго вручную). Тексты для разметки:")
for i, row in sample.iterrows():
    print(f"  [{i}] {row['text']}")

# Моя разметка вручную (долгое мероприятие)
my_labels = [
    22,  # [0]
    15,  # [1]
    59,  # [2]
    64,  # [3]
]
sample["my_label"] = my_labels
sample["match"] = sample["my_label"] == sample["label"]
show = sample[["text", "label_name", "my_label", "match"]].copy()
show["text"] = show["text"].str.slice(0, 70)
print()
print("Сравнение с истинными метками:")
print(show.to_string())
if any(my_labels):
    print(f"Совпало: {sample['match'].sum()} из {len(sample)}")
else:
    print("Ручная разметка ещё не заполнена (my_labels пустой).")
# ---- Конец кода ----



Считываем данные

{'text': 'I am still waiting on my card?', 'label': 12}
text          I am still waiting on my card?
label                                     12
label_name                      card_arrival
Name: 0, dtype: object
Размеры выборок:
  train: 10003 примеров
  test:  3080 примеров
  классов: 77

Все метки (id -> name):
   0  Refund_not_showing_up
   1  activate_my_card
   2  age_limit
   3  apple_pay_or_google_pay
   4  atm_support
   5  automatic_top_up
   6  balance_not_updated_after_bank_transfer
   7  balance_not_updated_after_cheque_or_cash_deposit
   8  beneficiary_not_allowed
   9  cancel_transfer
  10  card_about_to_expire
  11  card_acceptance
  12  card_arrival
  13  card_delivery_estimate
  14  card_linking
  15  card_not_working
  16  card_payment_fee_charged
  17  card_payment_not_recognised
  18  card_payment_wrong_exchange_rate
  19  card_swallowed
  20  cash_withdrawal_charge
  21  cash_withdrawal_not_recognised
  22  change_pin
  23  compromised_card
  2

### 5.2 Бейзлайн LLM разметка (7 баллов)

В этом пункте нужно получить бейзлайн разметку с помощью open source LLM и простого короткого промпта.

Для упрощения тут у нас уже есть golden set разметка (в случае если не было бы, то действовали как указано в лекции, или бы размечали для начала сами хотя бы 50-100 примеров), на которой мы можем проверять качество

Для старта используйте небольшую модель `Qwen/Qwen3-1.7B` с отключённым thinking (`enable_thinking=False`). Она запускается через Transformers на Apple Silicon (MPS, macOS 14+), NVIDIA GPU (CUDA) или CPU; на CPU работа может быть медленнее. Квантизация и bitsandbytes для этого варианта не нужны. Для быстрой отладки можно взять `Qwen/Qwen3-0.6B`; для итоговой оценки используйте модель, которая достигает заданного качества. Далее в следующих ячейках для улучшения можете использовать модели размера больше


In [ ]:
# Для базовой модели достаточно зависимостей из начала ноутбука.


In [6]:
import time

MODEL_NAME = "Qwen/Qwen3-1.7B"
MODEL_REVISION = "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
# На GPU уменьшаем память весов; на CPU используем float32.
model_dtype = torch.bfloat16 if device == "mps" else torch.float16 if device == "cuda" else torch.float32
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, revision=MODEL_REVISION, dtype=model_dtype,
).to(device).eval()
print(f"Модель: {MODEL_NAME}; устройство: {device}; dtype: {model_dtype}")
print("Для замены модели измените MODEL_NAME и задайте её revision либо None.")

# Модель и tokenizer готовы. Ниже реализуйте промпт, генерацию и парсинг.
# Для сообщений используйте tokenizer.apply_chat_template(..., enable_thinking=False).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Модель: Qwen/Qwen3-1.7B; устройство: cuda; dtype: torch.float16
Для замены модели измените MODEL_NAME и задайте её revision либо None.


Формируем простой короткий промпт, в котором укажем  все категории меток для разметки, и зададим нужный формат ответа: например, json {"label": str} или только название метки. Для выбранного формата напишем функцию парсинга


В примере решения разрешён также один Markdown-блок `json` вокруг JSON. Произвольный текст вокруг ответа не принимается; допустимый формат должен быть указан в промпте и одинаково проверяться во всех экспериментах.


In [7]:
# ---- Ваш код здесь ----
# Короткий промпт: список всех категорий и заданный формат ответа. По английски, а то QWen тупой.
prompt_template = (
    "You are a support-ticket classifier for a banking app.\n"
    "Assign the customer message to exactly one category from the list below.\n\n"
    "Categories:\n{labels}\n\n"
    "Answer ONLY with JSON of the form {{\"label\": \"<category>\"}} and nothing else.\n\n"
    "Message: {text}"
)

# ну не в текст же всё выписывать - подставим из ранее разобранного
LABELS_TEXT = "\n".join(label_names)
def build_prompt(text: str) -> str:
    return prompt_template.format(labels=LABELS_TEXT, text=text)
# ---- Конец кода ----


Делаем разметку 10-20 примеров, пишем функцию парсинга ответа (считаем метрику в скольких ответах нарушения следования формату), смотрим ответы
На выходе покажите таблицу из 10–20 строк: текст, истинная метка, предсказанная метка, исходный ответ и признак ошибки формата.


In [8]:
# Функция разметки вместе с промптом,

# ---- Ваш код здесь ----
print("""
    прокачиваем в цикле выбранную LLM для разметки данных через функцию annotate, добавляем разметку в исходный датасет и сохраняем в файл
    НЕ НАДО ЧИТАТЬ - ЭТО ПИСАЛ НЕ Я, А CLAUDE FABLE, я только местами правил (но всё осознал)
""")
LABEL_SET = set(label_names)
# Допускаем один markdown-блок ```json ... ``` вокруг JSON, больше ничего.
JSON_RE = re.compile(r"^\s*(?:```(?:json)?\s*)?(\{.*?\})\s*(?:```)?\s*$", re.DOTALL)


def parse_label(raw: str) -> tuple[str | None, int]:
    """Возвращает (label, corrupted). corrupted=1, если формат нарушен или метка не из списка."""
    m = JSON_RE.match(raw)
    if not m:
        return None, 1
    try:
        label = json.loads(m.group(1)).get("label")
    except (json.JSONDecodeError, AttributeError):
        return None, 1
    if not isinstance(label, str) or label.strip() not in LABEL_SET:
        return None, 1
    return label.strip(), 0


@torch.inference_mode()
def generate(prompt: str, max_new_tokens: int = 32, **gen_kwargs) -> str:
    """Один вызов модели: chat-шаблон без thinking.

    По умолчанию greedy-декодирование; gen_kwargs (do_sample, temperature, top_p)
    переопределяют его, это нужно для разметки с перекрытием.
    """
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, enable_thinking=False, return_tensors="pt",
    ).to(device)
    params = {"do_sample": False, **gen_kwargs}
    out = model.generate(
        input_ids, max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id, **params,
    )
    # Декодируем только новые токены, без промпта.
    return tokenizer.decode(out[0, input_ids.shape[1]:], skip_special_tokens=True).strip()


def annotate(text: str, prompt_fn=build_prompt, **gen_kwargs) -> tuple[str | None, int, str]:
    """Размечает один текст: (label, corrupted, raw_generation)."""
    raw = generate(prompt_fn(text), **gen_kwargs)
    label, corrupted = parse_label(raw)
    return label, corrupted, raw


def annotate_df(df: pd.DataFrame, prompt_fn=build_prompt, **gen_kwargs) -> pd.DataFrame:
    """Прогоняет annotate по датафрейму, добавляет колонки pred_label, corrupted, raw."""
    rows = []
    for text in tqdm(df["text"], desc="annotate"):
        label, corrupted, raw = annotate(text, prompt_fn, **gen_kwargs)
        rows.append({"pred_label": label, "corrupted": corrupted, "raw": raw})
    res = df.reset_index(drop=True).copy()
    return pd.concat([res, pd.DataFrame(rows)], axis=1)


# Фиксированная выборка для сравнения экспериментов (100 примеров из test).
# eval_df = test_df.sample(n=100, random_state=2024)
eval_df = test_df.sample(n=200, random_state=42)
# Отладочная выборка: 20 примеров из test, не пересекающихся с eval_df.
debug_df = test_df.drop(eval_df.index).sample(n=20, random_state=41)

t0 = time.time()
debug_res = annotate_df(debug_df)
print(f"Отладка: {len(debug_res)} примеров за {time.time() - t0:.0f} с")
print(f"Ошибок формата: {debug_res['corrupted'].sum()} из {len(debug_res)}")
debug_res["match"] = debug_res["pred_label"] == debug_res["label_name"]
show = debug_res[["text", "label_name", "pred_label", "match", "raw", "corrupted"]].copy()
show["text"] = show["text"].str.slice(0, 60)
show["raw"] = show["raw"].str.slice(0, 60)
print(show.to_string())
print(f"Совпало: {debug_res['match'].sum()} из {len(debug_res)}")
debug_res.to_csv("debug_baseline.csv", index=False)
# ---- Конец кода ----



    прокачиваем в цикле выбранную LLM для разметки данных через функцию annotate, добавляем разметку в исходный датасет и сохраняем в файл
    НЕ НАДО ЧИТАТЬ - ЭТО ПИСАЛ НЕ Я, А CLAUDE FABLE, я только местами правил (но всё осознал)



annotate:   0%|          | 0/20 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Отладка: 20 примеров за 6 с
Ошибок формата: 1 из 20
                                                            text                           label_name                           pred_label  match                                               raw  corrupted
0               I can't recall my passcode and need to reset it.                   passcode_forgotten                   passcode_forgotten   True                   {"label": "passcode_forgotten"}          0
1                     Why am I required to do an identity check?                  why_verify_identity                   verify_my_identity  False                   {"label": "verify_my_identity"}          0
2   My card was confiscated by an ATM. How do I get my card back                       card_swallowed                  lost_or_stolen_card  False                  {"label": "lost_or_stolen_card"}          0
3   There is a strange payment on my statement. What should I do          card_payment_not_recognised            transac

### 5.3 Оценка качества (2 балла)

Оцениваем качество разметки на тестовом датасет (либо на семпле из тестового датасета)


На выходе: accuracy, доля ошибок формата и число оценённых примеров. Непарсящиеся ответы считаем ошибками, а не исключаем из выборки.

Для сравнения вариантов используйте одни и те же 100 примеров из test с random_state=2024. 10–20 примеров оставьте для отладки формата. На такой небольшой выборке малые различия метрик не стоит переоценивать.


In [9]:
# ---- Ваш код здесь ----
print("""
    Инферим LLM на тесте, замеряем метрики
    НЕ НАДО ЧИТАТЬ - ЭТО ПИСАЛ НЕ Я, А CLAUDE FABLE, я только местами правил (но всё осознал)
""")
def evaluate(res: pd.DataFrame, name: str) -> dict:
    """Считает метрики качества разметки по результату annotate_df.

    res  - датафрейм с колонками label_name (истинная метка),
           pred_label (метка от LLM или None) и corrupted (0/1, ошибка формата).
    name - название эксперимента, попадёт в сводную таблицу.
    """
    # Число верных ответов. Если формат сломан, pred_label = None и сравнение
    # с label_name даёт False, то есть непарсящийся ответ автоматически
    # засчитывается как ошибка, а не выбрасывается из выборки.
    correct = (res["pred_label"] == res["label_name"]).sum()
    m = {
        "experiment": name,
        # Размер выборки: на 100 примерах шаг accuracy равен 0.01,
        # поэтому разницу в 1-3 пункта между экспериментами нельзя считать значимой.
        "n": len(res),
        # Доля верных ответов среди всех примеров, включая ошибки формата.
        "accuracy": round(correct / len(res), 3),
        # Доля ответов, которые не удалось распарсить или где метка не из списка.
        # Отдельная метрика: показывает, насколько модель следует инструкции,
        # а не насколько она понимает задачу.
        "format_error_rate": round(res["corrupted"].mean(), 3),
    }
    print(m)
    return m


# Список словарей с метриками каждого эксперимента (baseline, few-shot, CoT, ...).
# В конце работы превратим его в pd.DataFrame и покажем итоговую таблицу сравнения.
experiments = []

# Замеряем время: 100 вызовов модели могут занять заметное время,
# и это тоже аргумент при выборе модели и промпта для разметки.
t0 = time.time()
# Размечаем те же 100 примеров из test (eval_df, random_state=2024),
# Одна и та же выборка нужна, чтобы сравнивать промпты между собой.
baseline_res = annotate_df(eval_df)
print(f"Baseline: {len(baseline_res)} примеров за {time.time() - t0:.0f} с")
# Считаем метрики и добавляем строку в сводную таблицу экспериментов.
experiments.append(evaluate(baseline_res, "zero-shot baseline"))
# Сохраняем полный результат (текст, метки, сырой ответ модели), чтобы потом
# анализировать ошибки без повторного прогона модели.
baseline_res.to_csv("eval_baseline.csv", index=False)
# ---- Конец кода ----




    Инферим LLM на тесте, замеряем метрики
    НЕ НАДО ЧИТАТЬ - ЭТО ПИСАЛ НЕ Я, А CLAUDE FABLE, я только местами правил (но всё осознал)



annotate:   0%|          | 0/200 [00:00<?, ?it/s]

Baseline: 200 примеров за 59 с
{'experiment': 'zero-shot baseline', 'n': 200, 'accuracy': np.float64(0.49), 'format_error_rate': np.float64(0.04)}


### 6. Улучшение качества промпта

### 6.1 few shot prompt (4 баллов)

Добавим в промпт few-shot примеры (важно, чтобы не было data leak  c тестом): помогает исправлять поведение модели, когда описание в инструкции не справляется + модель лучше следует форматам
Вместо few-shot можно проверить другой способ улучшения инструкции: объясните выбор и сравните качество на тех же примерах.


In [10]:
# 6.1 few shot prompt
print("""
    НЕ НАДО ЧИТАТЬ - ЭТО ПИСАЛ НЕ Я, А CLAUDE FABLE, я только местами правил (но всё осознал)
""")

# ---- Ваш код здесь ----
# Простой few-shot: к каждой категории в списке добавляем один реальный пример
# сообщения из train. Идея: имена меток вроде card_arrival / card_delivery_estimate
# сами по себе неясны, а пример показывает границу класса. Берём train, а не test,
# чтобы не было утечки в eval_df.

# Один пример на класс: не слишком короткий и не слишком длинный (5-15 слов),
# чтобы промпт остался компактным.
_len = train_df["text"].str.split().str.len()
few_shot_pool = train_df[_len.between(5, 15)]
few_shot_examples = few_shot_pool.groupby("label_name").sample(n=1, random_state=1)
assert len(few_shot_examples) == len(label_names), "не для всех классов нашёлся пример"

# Блок категорий: label: "пример"
FEWSHOT_LABELS_TEXT = "\n".join(
    f'{row.label_name}: "{row.text.strip()}"'
    for row in few_shot_examples.sort_values("label_name").itertuples()
)

fewshot_template = (
    "You are a support-ticket classifier for a banking app.\n"
    "Assign the customer message to exactly one category from the list below.\n"
    "Each category is followed by one example message of that category.\n\n"
    "Categories:\n{labels}\n\n"
    "Answer ONLY with JSON of the form {{\"label\": \"<category>\"}} and nothing else.\n\n"
    "Message: {text}"
)


def build_prompt_fewshot(text: str) -> str:
    return fewshot_template.format(labels=FEWSHOT_LABELS_TEXT, text=text)


# Смотрим, насколько вырос промпт (в токенах) по сравнению с zero-shot.
n_zero = len(tokenizer(build_prompt("test"))["input_ids"])
n_few = len(tokenizer(build_prompt_fewshot("test"))["input_ids"])
print(f"Длина промпта в токенах: zero-shot={n_zero}, few-shot={n_few}")
print("Первые 5 строк блока категорий:")
print("\n".join(FEWSHOT_LABELS_TEXT.split("\n")[:5]))

# Разметка тех же 100 примеров eval_df и метрики.
t0 = time.time()
fewshot_res = annotate_df(eval_df, build_prompt_fewshot)
print(f"Few-shot: {len(fewshot_res)} примеров за {time.time() - t0:.0f} с")
experiments.append(evaluate(fewshot_res, "few-shot (1 example per label)"))

# Что изменилось относительно baseline на тех же примерах.
fewshot_res["baseline_pred"] = baseline_res["pred_label"].values
base_ok = fewshot_res["baseline_pred"] == fewshot_res["label_name"]
few_ok = fewshot_res["pred_label"] == fewshot_res["label_name"]
print(f"Исправлено ошибок baseline: {(~base_ok & few_ok).sum()}, "
      f"новых ошибок: {(base_ok & ~few_ok).sum()}, "
      f"изменилось меток всего: {(fewshot_res['baseline_pred'] != fewshot_res['pred_label']).sum()}")

# Примеры, где few-shot исправил ошибку baseline, и где сломал верный ответ.
cols = ["text", "label_name", "baseline_pred", "pred_label"]
print("\nИсправлено few-shot:")
print(fewshot_res.loc[~base_ok & few_ok, cols].head(5).to_string())
print("\nСломано few-shot:")
print(fewshot_res.loc[base_ok & ~few_ok, cols].head(5).to_string())

fewshot_res.to_csv("eval_fewshot.csv", index=False)
# ---- Конец кода ----


    НЕ НАДО ЧИТАТЬ - ЭТО ПИСАЛ НЕ Я, А CLAUDE FABLE, я только местами правил (но всё осознал)

Длина промпта в токенах: zero-shot=432, few-shot=1377
Первые 5 строк блока категорий:
Refund_not_showing_up: "I'm due a refund and it is not on my statement."
activate_my_card: "I just got my new card, how do I activate it?"
age_limit: "What is the age limit?"
apple_pay_or_google_pay: "how do I get top up to work for my card"
atm_support: "The card can be used at which ATMs?"


annotate:   0%|          | 0/200 [00:00<?, ?it/s]

Few-shot: 200 примеров за 80 с
{'experiment': 'few-shot (1 example per label)', 'n': 200, 'accuracy': np.float64(0.55), 'format_error_rate': np.float64(0.05)}
Исправлено ошибок baseline: 25, новых ошибок: 13, изменилось меток всего: 82

Исправлено few-shot:
                                                                          text                               label_name     baseline_pred                               pred_label
3   I just activated auto top-up, but it is not letting me enable it. Why not?                         automatic_top_up  activate_my_card                         automatic_top_up
4                        Why did I have to pay extra because I paid with card?                 card_payment_fee_charged     exchange_rate                 card_payment_fee_charged
6                I took out a foreign currency and the exchange rate is wrong.  wrong_exchange_rate_for_cash_withdrawal     exchange_rate  wrong_exchange_rate_for_cash_withdrawal
11                     $1 

### 6.2 chain-of-thoughts (3 балла)

Пробуем добавить сhain-of-thought в промпт: просим короткий reasoning и проверяем, помогает ли он на более сложных задачах. Объяснение модели не гарантирует правильность ответа и не обязательно отражает реальные причины решения.

Если используете JSON с reasoning, ответ может быть такого формата
{"reasoning": "why_this_class", "label": "one_of_the_categories"}
Вместо reasoning можно проверить другую гипотезу по ошибкам модели, отличающуюся от предыдущего пункта. Покажите изменение промпта, качество до/после и короткий вывод.


In [11]:
# 6.2 reasoning
print("""
    НЕ НАДО ЧИТАТЬ - ЭТО ПИСАЛ НЕ Я, А CLAUDE FABLE, я только местами правил (но всё осознал)
""")

# ---- Ваш код здесь ----
# Chain-of-thought: просим модель перед меткой написать одно короткое предложение
# с обоснованием. Формат ответа: {"reasoning": "...", "label": "..."}.
# Чтобы изолировать эффект reasoning, берём за основу zero-shot список категорий;
# переключатель COT_WITH_FEWSHOT=True строит reasoning поверх few-shot промпта из 6.1.
COT_WITH_FEWSHOT = False
cot_labels_block = FEWSHOT_LABELS_TEXT if COT_WITH_FEWSHOT else LABELS_TEXT

cot_template = (
    "You are a support-ticket classifier for a banking app.\n"
    "Assign the customer message to exactly one category from the list below.\n\n"
    "Categories:\n{labels}\n\n"
    "First think briefly: in ONE short sentence, state what the customer wants or what "
    "went wrong and why the chosen category fits better than similar ones. "
    "Then give the category.\n"
    "Answer ONLY with single-line JSON of the form "
    "{{\"reasoning\": \"<one sentence>\", \"label\": \"<category>\"}} and nothing else.\n\n"
    "Message: {text}"
)


def build_prompt_cot(text: str) -> str:
    return cot_template.format(labels=cot_labels_block, text=text)


def parse_cot(raw: str) -> tuple[str | None, int, str]:
    """Возвращает (label, corrupted, reasoning). Требования к формату те же, что в parse_label:
    один JSON, допускается markdown-блок json вокруг, метка должна быть из списка."""
    m = JSON_RE.match(raw)
    if not m:
        return None, 1, ""
    try:
        obj = json.loads(m.group(1), strict=False)  # strict=False: терпим перенос строки внутри reasoning
    except json.JSONDecodeError:
        return None, 1, ""
    if not isinstance(obj, dict):
        return None, 1, ""
    label = obj.get("label")
    reasoning = str(obj.get("reasoning", ""))
    if not isinstance(label, str) or label.strip() not in LABEL_SET:
        return None, 1, reasoning
    return label.strip(), 0, reasoning


def annotate_df_cot(df: pd.DataFrame, max_new_tokens: int = 128) -> pd.DataFrame:
    """Как annotate_df, но с reasoning-промптом, своим парсером и большим лимитом токенов."""
    rows = []
    for text in tqdm(df["text"], desc="annotate-cot"):
        raw = generate(build_prompt_cot(text), max_new_tokens=max_new_tokens)
        label, corrupted, reasoning = parse_cot(raw)
        rows.append({"pred_label": label, "corrupted": corrupted, "reasoning": reasoning, "raw": raw})
    res = df.reset_index(drop=True).copy()
    return pd.concat([res, pd.DataFrame(rows)], axis=1)


t0 = time.time()
cot_res = annotate_df_cot(eval_df)
print(f"CoT: {len(cot_res)} примеров за {time.time() - t0:.0f} с")
experiments.append(evaluate(cot_res, "CoT reasoning" + (" + few-shot" if COT_WITH_FEWSHOT else "")))

# Сравнение с baseline на тех же примерах.
cot_res["baseline_pred"] = baseline_res["pred_label"].values
base_ok = cot_res["baseline_pred"] == cot_res["label_name"]
cot_ok = cot_res["pred_label"] == cot_res["label_name"]
print(f"Исправлено ошибок baseline: {(~base_ok & cot_ok).sum()}, "
      f"новых ошибок: {(base_ok & ~cot_ok).sum()}, "
      f"изменилось меток всего: {(cot_res['baseline_pred'] != cot_res['pred_label']).sum()}")

# Reasoning полезен сам по себе для анализа ошибок: смотрим, как модель
# объясняет неверные ответы. Объяснение не гарантирует, что решение принято по этой причине.
pd.set_option("display.max_colwidth", 90)
cols = ["text", "label_name", "pred_label", "reasoning"]
print("\nОшибки CoT и объяснения модели:")
print(cot_res.loc[~cot_ok, cols].head(8).to_string())
print("\nОшибки формата CoT (сырой ответ):")
print(cot_res.loc[cot_res["corrupted"] == 1, ["text", "raw"]].head(5).to_string())

cot_res.to_csv("eval_cot.csv", index=False)
# ---- Конец кода ----


    НЕ НАДО ЧИТАТЬ - ЭТО ПИСАЛ НЕ Я, А CLAUDE FABLE, я только местами правил (но всё осознал)



annotate-cot:   0%|          | 0/200 [00:00<?, ?it/s]

CoT: 200 примеров за 264 с
{'experiment': 'CoT reasoning', 'n': 200, 'accuracy': np.float64(0.48), 'format_error_rate': np.float64(0.05)}
Исправлено ошибок baseline: 15, новых ошибок: 17, изменилось меток всего: 66

Ошибки CoT и объяснения модели:
                                                                          text                               label_name                pred_label                                                                                                                                                                                                                        reasoning
1                                  How do I retrieve my card from the machine?                           card_swallowed    card_delivery_estimate                                                                                                                  The customer is asking how to retrieve their card from a machine, which relates to card delivery and retrieval.
3   I just

### 6.3 Дальнейшие улучшения (6 баллов)

Далее улучшаем итеративно

Основные улучшения в общем случае происходит за счет:
- Аналитика ошибок,  в первую очередь анализируем ошибки разметки (только на трейне, чтобы не подогнаться под тест!), в том числе используя reasoning модели, чтобы понять причины. Также помогает спрашивать у самой модели и просить ее поправить начальный промпт/инструкцию

- Понимание бизнеса и домена, четкое описание в инструкции/промпте

Дополнительно, что тут может еще помочь:
- Упрощение задачи: размечать не одну, а несколько наиболее релеватных меток для каждого текста (=> растим recall)

- Использовать более "умные" LLM
- Размечаем с перекрытием: запускаем промпт n раз (например, 3) и агрегируем ответ. Улучшение: агрегурем результат ансамбля разных LLM (среди тех же размеров например: Qwen-3 8b) , можно с тем же промптом, либо промпты могут отлчичаться между собой few shot примерами

Тут нужно реализовать одно из улучшений (из лекции: слайды про улучшение 39-41, 46, либо списка выше, например, с перекрытием). Цель — получить итоговую accuracy ≥ 0.6 на тестовой выборке и сравнить её с исходным baseline.
Если текущая модель не достигает 0.6, проверьте более сильную модель — это допустимый способ улучшения. Если baseline уже выше 0.6, всё равно покажите осмысленный эксперимент и анализ: прирост от каждой отдельной техники не гарантирован. При смене модели укажите это в сравнении.


In [12]:
# ---- Ваш код здесь ----
print("""
    Промпты с улучшением качества (baseline: accuracy = 0.6), сравнение метрик, как каждое улучшение повляило
""")
print("""
    НЕ НАДО ЧИТАТЬ - ЭТО ПИСАЛ НЕ Я, А CLAUDE FABLE, я только местами правил (но всё осознал)
    попробуем анализ - что было не так. Затем Overlap, может помочь.
""")

# Анализ категорий ошибок: на каких истинных классах модель ошибается
# и с какими классами их путает. Для настройки промпта смотрим ошибки на train,
# чтобы не подогнать инструкцию под тестовые примеры.


def error_analysis(res: pd.DataFrame, top: int = 15) -> pd.DataFrame:
    """Таблица ошибок по истинным категориям, отсортированная по числу ошибок.

    Колонки: n (примеров класса в выборке), errors, error_rate,
    confused_with (в какие классы уходят ошибки, через запятую).
    """
    res = res.copy()
    res["is_error"] = res["pred_label"] != res["label_name"]
    # Ошибку формата показываем как отдельный "класс" FORMAT_ERROR.
    res["pred_shown"] = res["pred_label"].fillna("FORMAT_ERROR")

    per_class = (
        res.groupby("label_name")
        .agg(n=("is_error", "size"), errors=("is_error", "sum"))
        .assign(error_rate=lambda d: (d["errors"] / d["n"]).round(2))
    )
    # Для каждого класса собираем, куда именно уходят ошибки.
    confused = (
        res[res["is_error"]]
        .groupby("label_name")["pred_shown"]
        .agg(lambda x: ", ".join(f"{k}({v})" for k, v in x.value_counts().items()))
    )
    per_class["confused_with"] = confused
    per_class = per_class[per_class["errors"] > 0].sort_values(
        ["errors", "error_rate"], ascending=False
    )

    print(f"Классов с ошибками: {len(per_class)} из {res['label_name'].nunique()} в выборке")
    print(per_class.head(top).to_string())

    # Самые частые пары путаницы (истинная -> предсказанная).
    pairs = (
        res[res["is_error"]]
        .groupby(["label_name", "pred_shown"]).size()
        .sort_values(ascending=False).head(10)
    )
    print("\nСамые частые пары путаницы (истинная -> предсказанная):")
    print(pairs.to_string())
    return per_class


print("=== Ошибки baseline на eval_df (только для справки) ===")
error_analysis(baseline_res)

# Размечаем 100 примеров из train: именно эти ошибки используем для доработки промпта.
train_sample = train_df.sample(n=100, random_state=7)
train_res = annotate_df(train_sample)
evaluate(train_res, "zero-shot baseline (train sample)")
print("=== Ошибки baseline на train_sample ===")
train_errors = error_analysis(train_res)
train_res.to_csv("train_baseline.csv", index=False)

print("Не очень-то помогло, явной закономерности не видно")

# Разметка с перекрытием (self-consistency): запускаем тот же промпт n раз
# с сэмплированием и агрегируем ответы голосованием большинства.
# Greedy-декодирование даёт одинаковые ответы, поэтому нужна температура > 0.
# Побочный результат: доля согласных запусков (agreement) — это ещё одна мера
# уверенности модели, её можно сравнить с logprob-confidence в разделе 7.
from collections import Counter

N_RUNS = 3
# Рекомендованные Qwen3 параметры сэмплирования для режима без thinking.
SAMPLING = dict(do_sample=True, temperature=0.7, top_p=0.8)


def annotate_overlap(df: pd.DataFrame, n_runs: int = N_RUNS, prompt_fn=build_prompt) -> pd.DataFrame:
    """n_runs прогонов с сэмплированием + majority vote.

    Возвращает датафрейм с колонками run0..run{n-1} (метки каждого запуска),
    pred_label (итог голосования), agreement (доля запусков за итоговую метку),
    corrupted (1, если ни один запуск не дал валидную метку).
    """
    res = df.reset_index(drop=True).copy()
    for k in range(n_runs):
        torch.manual_seed(100 + k)  # разные seed => разные сэмплы
        run = annotate_df(df, prompt_fn, **SAMPLING)
        res[f"run{k}"] = run["pred_label"]

    run_cols = [f"run{k}" for k in range(n_runs)]

    def vote(row):
        valid = [x for x in row if pd.notna(x)]  # ошибки формата не голосуют
        if not valid:
            return None, 0.0
        label, cnt = Counter(valid).most_common(1)[0]  # при ничьей побеждает первый запуск
        return label, cnt / n_runs

    votes = res[run_cols].apply(vote, axis=1, result_type="expand")
    res["pred_label"], res["agreement"] = votes[0], votes[1]
    res["corrupted"] = res["pred_label"].isna().astype(int)
    return res


t0 = time.time()
overlap_res = annotate_overlap(eval_df)
print(f"Перекрытие: {N_RUNS} x {len(overlap_res)} примеров за {time.time() - t0:.0f} с")

# Качество каждого запуска по отдельности: видно разброс между сэмплами.
for k in range(N_RUNS):
    single = overlap_res[["label_name"]].assign(
        pred_label=overlap_res[f"run{k}"], corrupted=overlap_res[f"run{k}"].isna().astype(int)
    )
    evaluate(single, f"overlap run{k} (T=0.7)")
# Итог голосования — в сводную таблицу экспериментов.
experiments.append(evaluate(overlap_res, f"overlap majority n={N_RUNS}, T=0.7"))

# Сравнение с greedy baseline на тех же примерах: что именно изменилось.
overlap_res["baseline_pred"] = baseline_res["pred_label"].values
base_ok = overlap_res["baseline_pred"] == overlap_res["label_name"]
over_ok = overlap_res["pred_label"] == overlap_res["label_name"]
print(f"Исправлено ошибок baseline: {(~base_ok & over_ok).sum()}, "
      f"новых ошибок: {(base_ok & ~over_ok).sum()}, "
      f"изменилось меток всего: {(overlap_res['baseline_pred'] != overlap_res['pred_label']).sum()}")

# Accuracy в зависимости от согласованности запусков: если при полном согласии
# точность заметно выше, agreement годится как сигнал для отправки примера человеку.
overlap_res["match"] = over_ok
by_agr = (
    overlap_res.groupby("agreement")["match"]
    .agg(n="size", accuracy="mean").round(3)
    .sort_index(ascending=False)
)
print("Accuracy по уровню согласованности запусков:")
print(by_agr.to_string())

overlap_res.to_csv("eval_overlap.csv", index=False)


# ---- Конец кода ----


    Промпты с улучшением качества (baseline: accuracy = 0.6), сравнение метрик, как каждое улучшение повляило


    НЕ НАДО ЧИТАТЬ - ЭТО ПИСАЛ НЕ Я, А CLAUDE FABLE, я только местами правил (но всё осознал)
    попробуем анализ - что было не так. Затем Overlap, может помочь.

=== Ошибки baseline на eval_df (только для справки) ===
Классов с ошибками: 49 из 70 в выборке
                                  n  errors  error_rate                                                                                                                                       confused_with
label_name                                                                                                                                                                                                 
exchange_via_app                  7       7        1.00                                                                                                                exchange_rate(6), country_support(1)
order_physical_c

annotate:   0%|          | 0/100 [00:00<?, ?it/s]

{'experiment': 'zero-shot baseline (train sample)', 'n': 100, 'accuracy': np.float64(0.49), 'format_error_rate': np.float64(0.02)}
=== Ошибки baseline на train_sample ===
Классов с ошибками: 36 из 56 в выборке
                                         n  errors  error_rate                                                  confused_with
label_name                                                                                                                   
top_up_by_bank_transfer_charge           4       4        1.00  transfer_fee_charged(2), receiving_money(1), exchange_rate(1)
card_arrival                             3       3        1.00              card_delivery_estimate(2), lost_or_stolen_card(1)
direct_debit_payment_not_recognised      3       3        1.00                    declined_card_payment(2), request_refund(1)
beneficiary_not_allowed                  2       2        1.00                          transfer_not_received_by_recipient(2)
compromised_card                  

annotate:   0%|          | 0/200 [00:00<?, ?it/s]

annotate:   0%|          | 0/200 [00:00<?, ?it/s]

annotate:   0%|          | 0/200 [00:00<?, ?it/s]

Перекрытие: 3 x 200 примеров за 177 с
{'experiment': 'overlap run0 (T=0.7)', 'n': 200, 'accuracy': np.float64(0.49), 'format_error_rate': np.float64(0.045)}
{'experiment': 'overlap run1 (T=0.7)', 'n': 200, 'accuracy': np.float64(0.485), 'format_error_rate': np.float64(0.04)}
{'experiment': 'overlap run2 (T=0.7)', 'n': 200, 'accuracy': np.float64(0.48), 'format_error_rate': np.float64(0.04)}
{'experiment': 'overlap majority n=3, T=0.7', 'n': 200, 'accuracy': np.float64(0.485), 'format_error_rate': np.float64(0.04)}
Исправлено ошибок baseline: 0, новых ошибок: 1, изменилось меток всего: 12
Accuracy по уровню согласованности запусков:
             n  accuracy
agreement               
1.000000   178     0.539
0.666667    13     0.077
0.333333     1     0.000
0.000000     8     0.000


In [13]:
# 6.4 - ещё эксперименты

print("""
    Overlap в предыдущей ячейке не помог, надо делать другие эксперименты. Вернулся к ранее пропущенным 6.1 и 6.2.
    Здесь попробуем уменьшит количество классов первым запросом, и уточнить вторым.
""")

# ---- Ваш код здесь ----
# Двухэтапная классификация, версия 2.
#
# Что происходит: этап 1 выбирает тему сообщения из 8 тем, этап 2 выбирает метку
# только среди меток выбранных тем. Модели 1.7B проще выбирать из 8, а потом из 10-25,
# чем сразу из 77.
#
# Что изменилось после первого прогона (этап 1 давал 0.76, oracle этапа 2 - 0.79, итог 0.61):
# 1. Переразбиение групп. 10 из 24 ошибок этапа 1 были ошибками справочника, а не модели:
#    группа account_general была собрана "по остаточному принципу", а её метки по смыслу
#    относились к другим темам (apple_pay / supported_cards - это про пополнение,
#    visa_or_mastercard / country_support / card_acceptance - про получение карты).
#    Группа распущена, verify_* объединены с account в одну тему "аккаунт и верификация".
# 2. Топ-2 темы. Остальные ошибки - настоящие пересечения тем (declined_card_payment против
#    card_not_working, top_up_by_bank_transfer против transfer). Теперь этап 1 называет две
#    наиболее вероятные темы, а этап 2 выбирает из объединения их меток. Это приём
#    "размечать несколько релевантных меток => растим recall" из списка выше: цена та же,
#    два вызова на пример, а верная тема почти всегда попадает в топ-2.

GROUPS = {
    "card_get": (
        "getting, ordering, activating a card (physical, virtual, spare, disposable), card delivery, "
        "card expiry, linking a card, Visa vs Mastercard, where the card is accepted, supported countries",
        ["activate_my_card", "card_arrival", "card_delivery_estimate", "order_physical_card",
         "get_physical_card", "getting_spare_card", "getting_virtual_card",
         "get_disposable_virtual_card", "disposable_card_limits", "card_linking",
         "card_about_to_expire", "visa_or_mastercard", "card_acceptance", "country_support"],
    ),
    "card_problem": (
        "card or phone lost, stolen or compromised; card, contactless or virtual card not working; "
        "PIN or passcode problems",
        ["card_not_working", "virtual_card_not_working", "contactless_not_working",
         "lost_or_stolen_card", "lost_or_stolen_phone", "compromised_card", "pin_blocked",
         "change_pin", "passcode_forgotten"],
    ),
    "card_payment": (
        "a specific card payment, direct debit or purchase: declined, pending, not recognised, "
        "reverted, charged twice, extra fee or wrong exchange rate on a payment, refunds",
        ["card_payment_not_recognised", "card_payment_fee_charged",
         "card_payment_wrong_exchange_rate", "pending_card_payment", "reverted_card_payment?",
         "transaction_charged_twice", "extra_charge_on_statement",
         "direct_debit_payment_not_recognised", "request_refund", "Refund_not_showing_up",
         "declined_card_payment"],
    ),
    "cash_withdrawal": (
        "cash withdrawals and ATMs: where to withdraw, withdrawal declined, pending, not recognised, "
        "fee for withdrawal, wrong amount or wrong rate at the ATM, card swallowed by ATM",
        ["atm_support", "cash_withdrawal_charge", "cash_withdrawal_not_recognised",
         "declined_cash_withdrawal", "pending_cash_withdrawal", "wrong_amount_of_cash_received",
         "wrong_exchange_rate_for_cash_withdrawal", "card_swallowed"],
    ),
    "transfer": (
        "bank transfers between accounts: sending or receiving money, transfer pending, failed, "
        "declined, cancelled, not received, transfer fee, transfer timing, recipient not allowed",
        ["transfer_into_account", "transfer_not_received_by_recipient", "transfer_timing",
         "transfer_fee_charged", "pending_transfer", "failed_transfer", "declined_transfer",
         "cancel_transfer", "beneficiary_not_allowed", "receiving_money",
         "balance_not_updated_after_bank_transfer"],
    ),
    "top_up": (
        "topping up (adding money to) the account: by card, bank transfer, cash or cheque, "
        "Apple Pay or Google Pay; which cards and currencies can be used to top up; "
        "top-up failed, pending, reverted, limits, fees, automatic top-up, verifying a top-up",
        ["topping_up_by_card", "top_up_by_bank_transfer_charge", "top_up_by_card_charge",
         "top_up_by_cash_or_cheque", "top_up_failed", "top_up_limits", "top_up_reverted",
         "pending_top_up", "automatic_top_up", "verify_top_up",
         "balance_not_updated_after_cheque_or_cash_deposit",
         "apple_pay_or_google_pay", "supported_cards_and_currencies"],
    ),
    "exchange": (
        "currency exchange in general: exchange rates, exchange fees, exchanging via the app, "
        "which fiat currencies are supported",
        ["exchange_rate", "exchange_charge", "exchange_via_app", "fiat_currency_support"],
    ),
    "account_identity": (
        "the account itself and identity checks: editing personal details, closing the account, "
        "age limit, how or why to verify identity, verification failed, proving source of funds",
        ["edit_personal_details", "terminate_account", "age_limit",
         "verify_my_identity", "unable_to_verify_identity", "why_verify_identity",
         "verify_source_of_funds"],
    ),
}
TOP_K = 2  # сколько тем просим на этапе 1

# Проверка: каждая метка ровно в одной группе.
_all = [l for _, labs in GROUPS.values() for l in labs]
assert len(_all) == len(set(_all)), "метка попала в две группы"
assert set(_all) == set(label_names), set(label_names) ^ set(_all)
label2group = {l: g for g, (_, labs) in GROUPS.items() for l in labs}
GROUP_SET = set(GROUPS)
print("Размеры групп:", {g: len(labs) for g, (_, labs) in GROUPS.items()})

# Пример на метку из 6.1 (если ячейка 6.1 выполнена); внутри короткого списка примеры дёшевы.
label_example = (
    dict(zip(few_shot_examples["label_name"], few_shot_examples["text"]))
    if "few_shot_examples" in globals() else {}
)

# --- Этап 1: топ-2 темы ---
GROUPS_TEXT = "\n".join(f"{g}: {desc}" for g, (desc, _) in GROUPS.items())
stage1_template = (
    "You are a support-ticket classifier for a banking app.\n"
    "Choose the {k} topics that most likely match the customer message, most likely first.\n\n"
    "Topics:\n{groups}\n\n"
    "Answer ONLY with JSON of the form {{\"groups\": [\"<topic1>\", \"<topic2>\"]}} and nothing else.\n\n"
    "Message: {text}"
)

# --- Этап 2: метка среди меток выбранных тем ---
stage2_template = (
    "You are a support-ticket classifier for a banking app.\n"
    "The customer message is about one of these topics: {group_desc}.\n"
    "Assign it to exactly one category from the list below{examples_note}.\n\n"
    "Categories:\n{labels}\n\n"
    "Answer ONLY with JSON of the form {{\"label\": \"<category>\"}} and nothing else.\n\n"
    "Message: {text}"
)


def build_prompt_stage1(text: str) -> str:
    return stage1_template.format(k=TOP_K, groups=GROUPS_TEXT, text=text)


def build_prompt_stage2(text: str, groups: list[str]) -> str:
    labs = [l for g in groups for l in GROUPS[g][1]]  # метки первой темы идут первыми
    lines = [f'{l}: "{label_example[l].strip()}"' if l in label_example else l for l in labs]
    return stage2_template.format(
        group_desc="; or ".join(GROUPS[g][0] for g in groups),
        labels="\n".join(lines), text=text,
        examples_note=" (each with one example message)" if label_example else "",
    )


def parse_groups(raw: str) -> tuple[list[str], int]:
    """Этап 1: JSON с полем groups - список тем. Оставляем валидные, без повторов, не более TOP_K.
    corrupted=1, если валидных тем нет."""
    m = JSON_RE.match(raw)
    if not m:
        return [], 1
    try:
        val = json.loads(m.group(1)).get("groups")
    except (json.JSONDecodeError, AttributeError):
        return [], 1
    if isinstance(val, str):
        val = [val]
    if not isinstance(val, list):
        return [], 1
    groups = []
    for g in val:
        if isinstance(g, str) and g.strip() in GROUP_SET and g.strip() not in groups:
            groups.append(g.strip())
    return groups[:TOP_K], (0 if groups else 1)


def parse_json_field(raw: str, key: str, allowed: set) -> tuple[str | None, int]:
    """Этап 2: один JSON, поле key со значением из allowed."""
    m = JSON_RE.match(raw)
    if not m:
        return None, 1
    try:
        val = json.loads(m.group(1)).get(key)
    except (json.JSONDecodeError, AttributeError):
        return None, 1
    if not isinstance(val, str) or val.strip() not in allowed:
        return None, 1
    return val.strip(), 0


def annotate_df_two_stage(df: pd.DataFrame, known_groups=None) -> pd.DataFrame:
    """Два вызова на пример. known_groups: последовательность тем той же длины (строка,
    список тем или None) - тогда этап 1 пропускается и берутся эти темы. Так делаем
    oracle (верная тема) и вариант top-1 (переиспользуем ответы этапа 1)."""
    rows = []
    use_known = known_groups is not None
    known = list(known_groups) if use_known else [None] * len(df)
    for text, kg in tqdm(zip(df["text"], known), total=len(df), desc="two-stage"):
        if use_known:
            raw1 = ""
            groups = [kg] if isinstance(kg, str) else list(kg or [])
        else:
            raw1 = generate(build_prompt_stage1(text))
            groups, c1 = parse_groups(raw1)
        if not groups:  # темы не распознаны - итог сразу ошибка формата
            rows.append({"pred_groups": [], "pred_label": None, "corrupted": 1, "raw1": raw1, "raw2": ""})
            continue
        raw2 = generate(build_prompt_stage2(text, groups))
        allowed = {l for g in groups for l in GROUPS[g][1]}
        label, c2 = parse_json_field(raw2, "label", allowed)
        rows.append({"pred_groups": groups, "pred_label": label, "corrupted": c2, "raw1": raw1, "raw2": raw2})
    res = df.reset_index(drop=True).copy()
    res["true_group"] = res["label_name"].map(label2group)
    res = pd.concat([res, pd.DataFrame(rows)], axis=1)
    res["group_top1"] = res["pred_groups"].map(lambda g: g[0] if g else None)
    res["group_hit"] = [tg in pg for tg, pg in zip(res["true_group"], res["pred_groups"])]
    return res


# Полный двухэтапный прогон на тех же 100 примерах.
t0 = time.time()
two_res = annotate_df_two_stage(eval_df)
print(f"Two-stage v2: {len(two_res)} примеров за {time.time() - t0:.0f} с")
print(f"Этап 1: верная тема первой - {(two_res['group_top1'] == two_res['true_group']).mean():.2f}, "
      f"верная тема в топ-{TOP_K} - {two_res['group_hit'].mean():.2f}, "
      f"ошибок формата на этапе 1: {(two_res['pred_groups'].map(len) == 0).sum()}")
# В сводную таблицу experiments идёт только полная двухэтапная разметка (этап 1 + этап 2).
experiments.append(evaluate(two_res, f"two-stage v2: top-{TOP_K} groups -> label"))

# Диагностика (в сводную таблицу НЕ входит): этап 2 с единственной верной темой.
# Разница между oracle и итогом = цена ошибок маршрутизации на этапе 1.
t0 = time.time()
oracle_res = annotate_df_two_stage(eval_df, known_groups=eval_df["label_name"].map(label2group))
print(f"Oracle stage 2: {len(oracle_res)} примеров за {time.time() - t0:.0f} с")
print("[диагностика, не в таблице] ", end="")
evaluate(oracle_res, "stage 2 only, true group given (oracle)")

# Вариант top-1: та же схема, но этап 2 видит только первую тему из ответа этапа 1.
# Ответы этапа 1 переиспользуем из two_res, поэтому это ещё 100 вызовов, а не 200.
# Разница с top-2 показывает, сколько даёт страховка второй темой.
t0 = time.time()
top1_res = annotate_df_two_stage(eval_df, known_groups=two_res["group_top1"])
print(f"Two-stage top-1: {len(top1_res)} примеров за {time.time() - t0:.0f} с")
experiments.append(evaluate(top1_res, "two-stage v2: top-1 group -> label"))
top1_ok = top1_res["pred_label"] == top1_res["label_name"]
top2_ok = two_res["pred_label"] == two_res["label_name"]
print(f"top-2 против top-1 на тех же примерах: исправлено {(~top1_ok & top2_ok).sum()}, "
      f"сломано {(top1_ok & ~top2_ok).sum()}, изменилось меток {(top1_res['pred_label'] != two_res['pred_label']).sum()}")
# Где вторая тема спасла: верная тема была второй, и top-2 ответил верно.
saved = two_res[(two_res["group_top1"] != two_res["true_group"]) & two_res["group_hit"] & top2_ok]
print(f"Верная тема была второй и метка угадана: {len(saved)} примеров")
# Где вторая тема навредила: тема верная первой, top-1 угадал, top-2 ушёл в метку из второй темы.
hurt = two_res[(two_res["group_top1"] == two_res["true_group"]) & top1_ok & ~top2_ok]
print(f"Первая тема верная, но лишние метки второй темы сбили модель: {len(hurt)} примеров")
if len(hurt):
    print(hurt[["text", "label_name", "pred_label"]].assign(text=lambda d: d["text"].str.slice(0, 60)).to_string())
top1_res.to_csv("eval_two_stage_top1.csv", index=False)

# Сравнение с baseline на тех же примерах.
two_res["baseline_pred"] = baseline_res["pred_label"].values
base_ok = two_res["baseline_pred"] == two_res["label_name"]
two_ok = two_res["pred_label"] == two_res["label_name"]
print(f"Исправлено ошибок baseline: {(~base_ok & two_ok).sum()}, "
      f"новых ошибок: {(base_ok & ~two_ok).sum()}, "
      f"изменилось меток всего: {(two_res['baseline_pred'] != two_res['pred_label']).sum()}")

# Промахи этапа 1: верной темы нет даже в топ-2.
miss = two_res[~two_res["group_hit"]]
print(f"\nПромахи этапа 1 (верной темы нет в топ-{TOP_K}): {len(miss)}")
print(miss[["text", "true_group", "pred_groups"]].assign(text=lambda d: d["text"].str.slice(0, 60)).to_string())

# Ошибки этапа 2 при верной теме в списке: путаница внутри темы.
inside = two_res[two_res["group_hit"] & ~two_ok]
print(f"\nОшибки этапа 2 при верной теме в топ-{TOP_K}: {len(inside)}")
print(inside[["label_name", "pred_label"]].value_counts().head(10).to_string())

two_res.to_csv("eval_two_stage.csv", index=False)

# Сводная таблица экспериментов на текущий момент: только полноценные схемы разметки.
print("\nСводная таблица экспериментов (eval_df, n=100):")
print(pd.DataFrame(experiments).drop_duplicates("experiment", keep="last").to_string(index=False))
# ---- Конец кода ----


    Overlap в предыдущей ячейке не помог, надо делать другие эксперименты. Вернулся к ранее пропущенным 6.1 и 6.2.
    Здесь попробуем уменьшит количество классов первым запросом, и уточнить вторым.

Размеры групп: {'card_get': 14, 'card_problem': 9, 'card_payment': 11, 'cash_withdrawal': 8, 'transfer': 11, 'top_up': 13, 'exchange': 4, 'account_identity': 7}


two-stage:   0%|          | 0/200 [00:00<?, ?it/s]

Two-stage v2: 200 примеров за 129 с
Этап 1: верная тема первой - 0.76, верная тема в топ-2 - 0.84, ошибок формата на этапе 1: 0
{'experiment': 'two-stage v2: top-2 groups -> label', 'n': 200, 'accuracy': np.float64(0.595), 'format_error_rate': np.float64(0.02)}


two-stage:   0%|          | 0/200 [00:00<?, ?it/s]

Oracle stage 2: 200 примеров за 57 с
[диагностика, не в таблице] {'experiment': 'stage 2 only, true group given (oracle)', 'n': 200, 'accuracy': np.float64(0.77), 'format_error_rate': np.float64(0.005)}


two-stage:   0%|          | 0/200 [00:00<?, ?it/s]

Two-stage top-1: 200 примеров за 58 с
{'experiment': 'two-stage v2: top-1 group -> label', 'n': 200, 'accuracy': np.float64(0.595), 'format_error_rate': np.float64(0.01)}
top-2 против top-1 на тех же примерах: исправлено 13, сломано 13, изменилось меток 55
Верная тема была второй и метка угадана: 10 примеров
Первая тема верная, но лишние метки второй темы сбили модель: 13 примеров
                                                             text                        label_name                      pred_label
20   Hi, I have been overcharged for my payment last Saturday. I   card_payment_wrong_exchange_rate       extra_charge_on_statement
26                            What type of ATMs accept this card?                       atm_support                 card_acceptance
34                                 I couldn't get cash at the atm          declined_cash_withdrawal                     atm_support
81               How can I speed up a transfer?  Mine is pending.                  pendi

In [19]:
# 6.5 - доработка обоих этапов по ошибкам на train

# ---- Ваш код здесь ----
# Итоги прошлой версии 6.5: текстовые правила ("копируй имя точно", "выбирай специфичную") дали
# минус 2 пункта и больше ошибок формата - модель 1.7B им не следует. Кодбук дал плюс там, где он
# был (0.51 -> 0.59 на метках из кодбука), но покрывал 16 меток из 77 и не содержал главных пар
# из таблицы на 300 примерах train. Случайные примеры из 6.1 местами вредят: у get_physical_card
# пример про PIN, у order_physical_card про адрес доставки.
#
# Что меняется здесь (все наблюдения только с train):
#  Этап 2: правил нет; кодбук расширен до 37 меток по таблице train-300; случайный пример
#          показывается только для меток, у которых нет строки кодбука.
#  Этап 1: описания тем переписаны под путаницу тем на train-300 (card_payment уходит в transfer
#          и exchange, card_get в card_problem, top_up в transfer) + по 3 примера на тему из train.
# Эффекты измеряем раздельно: этап 2 v3 со старыми темами, этап 1 v3 отдельно, затем оба вместе.

# --- 1. Анализ ошибок на train (300 примеров); если уже посчитан в прошлом запуске, не повторяем ---
TRAIN_N = 300
if "train_big_res" not in globals() or len(train_big_res) != TRAIN_N:
    train_big = train_df.sample(n=TRAIN_N, random_state=11)
    t0 = time.time()
    train_big_res = annotate_df(train_big)
    print(f"Train {TRAIN_N} примеров за {time.time() - t0:.0f} с")
    train_big_res.to_csv("train_baseline_300.csv", index=False)
print("[диагностика, не в таблице] ", end="")
evaluate(train_big_res, f"zero-shot baseline (train, n={TRAIN_N})")
err = train_big_res[train_big_res["pred_label"].notna() & (train_big_res["pred_label"] != train_big_res["label_name"])]
pairs = err.groupby(["label_name", "pred_label"]).size().sort_values(ascending=False)
print("Пары путаницы меток (истинная -> предсказанная), частота >= 2:")
print(pairs[pairs >= 2].to_string())
# Путаница тем: zero-shot метку переводим в тему и сравниваем с темой истинной метки.
_v = train_big_res[train_big_res["pred_label"].notna()].assign(
    tg=lambda d: d["label_name"].map(label2group), pg=lambda d: d["pred_label"].map(label2group))
gpairs = _v[_v["tg"] != _v["pg"]].groupby(["tg", "pg"]).size().sort_values(ascending=False)
print(f"\nДоля верной темы по zero-shot метке (train): {(_v['tg'] == _v['pg']).mean():.3f}")
print("Путаница тем (истинная -> предсказанная), частота >= 2:")
print(gpairs[gpairs >= 2].to_string())

# --- 2. Кодбук: 37 меток по парам с частотой >= 2 на train-300 ---
CODEBOOK = {
    # физическая карта: тройка order / get / delivery_estimate + arrival
    "order_physical_card": "user explicitly wants to ORDER or be sent a physical card",
    "get_physical_card": "general questions about obtaining a physical card: how it works, what comes with it (PIN), where it is sent",
    "card_arrival": "the ordered card has NOT arrived yet; user is waiting, asks where it is or for tracking",
    "card_delivery_estimate": "user asks HOW LONG delivery takes or when to expect the card (no complaint that it is late)",
    "card_about_to_expire": "the card is about to expire; what happens, will a new one be sent",
    "getting_virtual_card": "how to get or create a virtual card",
    "virtual_card_not_working": "an existing virtual card is rejected or does not work",
    "get_disposable_virtual_card": "how to get or create a DISPOSABLE virtual card",
    "disposable_card_limits": "LIMITS on disposable cards: how many, how often, max amount",
    "age_limit": "minimum age to open an account or get a card",
    # проблемы с картой
    "compromised_card": "user suspects fraud or leaked card details, unknown activity; the card itself is not lost",
    "lost_or_stolen_card": "the physical card itself is lost or stolen",
    # платежи картой и списания
    "declined_card_payment": "user TRIED to pay by card and the payment was declined or did not go through",
    "card_payment_not_recognised": "user SEES a card payment on the statement that they did not make",
    "direct_debit_payment_not_recognised": "an unknown DIRECT DEBIT (recurring payment) appeared; not a card purchase",
    "pending_card_payment": "a card purchase still shows as PENDING, when will it complete",
    "card_payment_fee_charged": "a FEE was added to a card payment",
    "card_payment_wrong_exchange_rate": "the exchange rate applied to a CARD PAYMENT was wrong",
    "extra_charge_on_statement": "an unexpected extra charge or fee on the statement, not tied to exchange or a known fee",
    # наличные
    "wrong_exchange_rate_for_cash_withdrawal": "the exchange rate applied to a CASH WITHDRAWAL at an ATM was wrong",
    "wrong_amount_of_cash_received": "the ATM gave LESS or a different amount of cash than requested",
    "cash_withdrawal_not_recognised": "a cash withdrawal on the statement that the user did NOT make",
    "declined_cash_withdrawal": "the ATM DECLINED the withdrawal",
    "pending_cash_withdrawal": "a cash withdrawal still shows as PENDING",
    # переводы
    "failed_transfer": "the transfer FAILED, was rejected or returned",
    "transfer_not_received_by_recipient": "the transfer was sent, but the recipient has not got the money",
    "pending_transfer": "a transfer still shows as PENDING",
    "beneficiary_not_allowed": "the app does not allow to add or send money to a particular recipient",
    "transfer_fee_charged": "a fee for SENDING money to someone else (outgoing transfer)",
    "receiving_money": "receiving money, salary or payments INTO the account, including in other currencies",
    # пополнение и обмен
    "top_up_by_bank_transfer_charge": "a FEE for topping up the own account by bank transfer",
    "top_up_by_card_charge": "a FEE for topping up the own account BY CARD",
    "top_up_by_cash_or_cheque": "how to top up with CASH or a CHEQUE",
    "exchange_rate": "general question about exchange rates: which rate is used, where to see it",
    "exchange_charge": "a FEE for exchanging currency",
    # верификация
    "why_verify_identity": "user asks WHY identity verification is required",
    "verify_my_identity": "user asks HOW to verify identity, which documents are needed",
}
assert set(CODEBOOK) <= LABEL_SET, set(CODEBOOK) - LABEL_SET
print(f"\nКодбук: {len(CODEBOOK)} меток")

# --- 3. Этап 2 v3: без правил; кодбук, а для остальных меток пример из 6.1 ---
stage2_v3_template = (
    "You are a support-ticket classifier for a banking app.\n"
    "The customer message is about: {group_desc}.\n"
    "Assign it to exactly one category from the list below. "
    "A category may be followed by a short note or an example message.\n\n"
    "Categories:\n{labels}\n\n"
    "Answer ONLY with JSON of the form {{\"label\": \"<category>\"}} and nothing else.\n\n"
    "Message: {text}"
)


def build_prompt_stage2_v3(text: str, groups: list[str]) -> str:
    labs = [l for g in groups for l in GROUPS[g][1]]
    lines = []
    for l in labs:
        if l in CODEBOOK:
            lines.append(f"{l}: {CODEBOOK[l]}")
        elif l in label_example:
            lines.append(f'{l}: e.g. "{label_example[l].strip()}"')
        else:
            lines.append(l)
    return stage2_v3_template.format(
        group_desc="; or ".join(GROUP_DESC_V3.get(g, GROUPS[g][0]) for g in groups),
        labels="\n".join(lines), text=text,
    )


# --- 4. Этап 1 v3: описания тем под путаницу на train + 3 примера на тему из train ---
GROUP_DESC_V3 = {
    "card_get": "getting a NEW card: ordering, delivery, tracking, activation, PIN arriving with the card, "
                "expiry, virtual / spare / disposable cards, Visa vs Mastercard, supported countries, where the card is accepted",
    "card_problem": "an EXISTING card or phone: lost, stolen or compromised; card, contactless or virtual card rejected "
                    "or not working; PIN blocked, PIN change, passcode forgotten",
    "card_payment": "a card PURCHASE or DIRECT DEBIT on the statement: declined, pending, not recognised, reverted, "
                    "charged twice, extra fee or wrong exchange rate on that payment, refunds for purchases. "
                    "NOT bank transfers, NOT general exchange-rate questions",
    "cash_withdrawal": "CASH and ATMs: where to withdraw, withdrawal declined, pending or not recognised, "
                       "withdrawal fee, wrong amount or wrong rate at the ATM, card swallowed by the ATM",
    "transfer": "a BANK TRANSFER to or from another person or account: sending, receiving, pending, failed, "
                "declined, cancelled, not received, transfer fee, timing, recipient not allowed. "
                "NOT card purchases, NOT topping up the own account",
    "top_up": "adding money to the user's OWN account (top-up): by card, bank transfer, cash, cheque, Apple/Google Pay; "
              "which cards or currencies can be used; top-up failed, pending, reverted, limits, fees, automatic top-up",
    "exchange": "GENERAL questions about currency exchange: rates, exchange fees, how to exchange in the app, "
                "supported fiat currencies. NOT a complaint about a specific payment or withdrawal",
    "account_identity": "the account itself and identity checks: personal details, closing the account, age limit, "
                        "how or why to verify identity, verification failed, source of funds",
}
assert set(GROUP_DESC_V3) == set(GROUPS)

# 3 примера на тему из train: короткие тексты, фиксированный seed. Печатаем, чтобы проверить глазами.
_pool = train_df.assign(g=train_df["label_name"].map(label2group))
_pool = _pool[_pool["text"].str.split().str.len().between(6, 12)]
group_examples = {g: list(_pool[_pool["g"] == g].sample(n=3, random_state=3)["text"].str.strip()) for g in GROUPS}
print("\nПримеры для тем (этап 1):")
for g, exs in group_examples.items():
    print(f"  {g}: " + " | ".join(f'"{e}"' for e in exs))

GROUPS_TEXT_V3 = "\n".join(
    f"{g}: {GROUP_DESC_V3[g]}\n    examples: " + "; ".join(f'"{e}"' for e in group_examples[g])
    for g in GROUPS
)
stage1_v3_template = (
    "You are a support-ticket classifier for a banking app.\n"
    "Choose the single topic that best matches the customer message. "
    "Each topic has a description and example messages.\n\n"
    "Topics:\n{groups}\n\n"
    "Answer ONLY with JSON of the form {{\"group\": \"<topic>\"}} and nothing else.\n\n"
    "Message: {text}"
)


def build_prompt_stage1_v3(text: str) -> str:
    return stage1_v3_template.format(groups=GROUPS_TEXT_V3, text=text)


print(f"\nДлина промпта этапа 1 в токенах: v2={len(tokenizer(build_prompt_stage1('x'))['input_ids'])}, "
      f"v3={len(tokenizer(build_prompt_stage1_v3('x'))['input_ids'])}")
print("--- пример промпта этапа 2 v3 для темы card_get (первые 900 символов, полный промпт не обрезается) ---")
print(build_prompt_stage2_v3("Is my card on the way?", ["card_get"])[:900])


def run_stage1(df: pd.DataFrame, prompt_builder) -> pd.Series:
    """Этап 1: одна тема на пример; None при ошибке формата."""
    out = []
    for text in tqdm(df["text"], desc="stage-1"):
        g, _ = parse_json_field(generate(prompt_builder(text)), "group", GROUP_SET)
        out.append(g)
    return pd.Series(out, index=range(len(df)))


def run_stage2(df: pd.DataFrame, groups_series, prompt_builder) -> pd.DataFrame:
    """Этап 2 с готовыми темами: один вызов на пример."""
    rows = []
    for text, g in tqdm(zip(df["text"], groups_series), total=len(df), desc="stage-2"):
        groups = [g] if isinstance(g, str) else list(g or [])
        if not groups:
            rows.append({"pred_label": None, "corrupted": 1, "raw": ""})
            continue
        raw = generate(prompt_builder(text, groups))
        allowed = {l for gg in groups for l in GROUPS[gg][1]}
        label, corrupted = parse_json_field(raw, "label", allowed)
        rows.append({"pred_label": label, "corrupted": corrupted, "raw": raw})
    res = pd.concat([df.reset_index(drop=True), pd.DataFrame(rows)], axis=1)
    res["pred_group"] = list(groups_series)
    return res


# --- 5. Прогоны на eval_df ---
true_groups = eval_df["label_name"].map(label2group).reset_index(drop=True)
t0 = time.time()
# (a) этап 2 v3 со старыми темами top-1: чистый эффект кодбука
s2_old_res = run_stage2(eval_df, two_res["group_top1"], build_prompt_stage2_v3)
# (b) этап 1 v3: точность выбора темы против v2
groups_v3 = run_stage1(eval_df, build_prompt_stage1_v3)
# (c) оба этапа v3
v3_res = run_stage2(eval_df, groups_v3, build_prompt_stage2_v3)
print(f"\nТри прогона по {len(eval_df)} за {time.time() - t0:.0f} с")

acc_g_v2 = (two_res["group_top1"] == true_groups).mean()
acc_g_v3 = (groups_v3 == true_groups).mean()
print(f"Этап 1, верная тема: v2 = {acc_g_v2:.3f}, v3 = {acc_g_v3:.3f}, "
      f"ошибок формата v3: {int(groups_v3.isna().sum())}")
gp = pd.DataFrame({"true": true_groups, "pred": groups_v3})
gp = gp[gp["true"] != gp["pred"]].groupby(["true", "pred"], dropna=False).size().sort_values(ascending=False)
print("Оставшаяся путаница тем v3 (истинная -> предсказанная):")
print(gp.head(8).to_string())

print("\nОтправная точка: ", end="")
evaluate(top1_res, "two-stage v2: top-1 group -> label")
experiments.append(evaluate(s2_old_res, "two-stage v3: stage1 v2 + stage2 codebook"))
experiments.append(evaluate(v3_res, "two-stage v3: stage1 v3 + stage2 codebook"))

base_ok = top1_res["pred_label"] == top1_res["label_name"]
for name, res in [("stage2 codebook", s2_old_res), ("both v3", v3_res)]:
    ok = res["pred_label"] == res["label_name"]
    print(f"{name}: исправлено {(~base_ok & ok).sum()}, сломано {(base_ok & ~ok).sum()}, "
          f"ошибок формата {int(top1_res['corrupted'].sum())} -> {int(res['corrupted'].sum())}")
in_cb = v3_res["label_name"].isin(CODEBOOK)
print(f"Примеры с меткой из кодбука: {int(in_cb.sum())}; accuracy top-1 = {base_ok[in_cb].mean():.2f}, "
      f"stage2 codebook = {(s2_old_res['pred_label'] == s2_old_res['label_name'])[in_cb].mean():.2f}")
# Потолок: точность этапа 2 там, где тема верная.
right = v3_res["pred_group"] == true_groups
print(f"Этап 2 v3 при верной теме: accuracy = {(v3_res['pred_label'] == v3_res['label_name'])[right].mean():.2f} "
      f"на {int(right.sum())} примерах")

v3_res.to_csv("eval_two_stage_v3.csv", index=False)
print("\nСводная таблица экспериментов (eval_df, n=%d):" % len(eval_df))
print(pd.DataFrame(experiments).drop_duplicates("experiment", keep="last").to_string(index=False))
# ---- Конец кода ----

[диагностика, не в таблице] {'experiment': 'zero-shot baseline (train, n=300)', 'n': 300, 'accuracy': np.float64(0.527), 'format_error_rate': np.float64(0.04)}
Пары путаницы меток (истинная -> предсказанная), частота >= 2:
label_name                               pred_label                        
order_physical_card                      get_physical_card                     4
                                         card_delivery_estimate                4
card_arrival                             card_delivery_estimate                3
failed_transfer                          transfer_not_received_by_recipient    3
extra_charge_on_statement                exchange_rate                         3
card_about_to_expire                     getting_virtual_card                  2
card_arrival                             lost_or_stolen_card                   2
disposable_card_limits                   get_disposable_virtual_card           2
declined_cash_withdrawal                 cash_withdra

stage-2:   0%|          | 0/200 [00:00<?, ?it/s]

stage-1:   0%|          | 0/200 [00:00<?, ?it/s]

stage-2:   0%|          | 0/200 [00:00<?, ?it/s]


Три прогона по 200 за 172 с
Этап 1, верная тема: v2 = 0.760, v3 = 0.805, ошибок формата v3: 0
Оставшаяся путаница тем v3 (истинная -> предсказанная):
true          pred            
card_payment  exchange            6
transfer      card_payment        5
top_up        transfer            4
card_payment  card_problem        3
transfer      top_up              3
card_problem  card_get            3
top_up        exchange            2
card_get      account_identity    1

Отправная точка: {'experiment': 'two-stage v2: top-1 group -> label', 'n': 200, 'accuracy': np.float64(0.595), 'format_error_rate': np.float64(0.01)}
{'experiment': 'two-stage v3: stage1 v2 + stage2 codebook', 'n': 200, 'accuracy': np.float64(0.625), 'format_error_rate': np.float64(0.005)}
{'experiment': 'two-stage v3: stage1 v3 + stage2 codebook', 'n': 200, 'accuracy': np.float64(0.635), 'format_error_rate': np.float64(0.005)}
stage2 codebook: исправлено 11, сломано 5, ошибок формата 2 -> 1
both v3: исправлено 17, сломано 

In [20]:
# 6.6 - модель побольше: Qwen3-4B на той же схеме

# ---- Ваш код здесь ----
# Qwen3-4B в fp16 занимает ~8 ГБ весов; 1.7B перед загрузкой выгружаем, иначе в 16 ГБ будет тесно.
# Qwen3-8B в fp16 не помещается (~16 ГБ весов), без квантизации его не пробуем.
# Прогоняем два варианта: zero-shot baseline и лучшую схему (two-stage v3), чтобы сравнить
# с 1.7B на тех же примерах. Смена модели указана в имени эксперимента.
# ВНИМАНИЕ: после этой ячейки в памяти остаётся 4B; разделы 7-8 будут считаться на ней.
# Чтобы вернуть 1.7B, перезапустите ячейку загрузки модели в разделе 5.2.
import gc

RUN_4B = True
BIG_MODEL = "Qwen/Qwen3-4B"

if RUN_4B:
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print(f"Загружаем {BIG_MODEL} ...")
    tokenizer = AutoTokenizer.from_pretrained(BIG_MODEL)
    model = AutoModelForCausalLM.from_pretrained(BIG_MODEL, dtype=model_dtype).to(device).eval()
    if device == "cuda":
        print(f"VRAM занято: {torch.cuda.memory_allocated() / 2**30:.1f} ГБ")

    t0 = time.time()
    base4_res = annotate_df(eval_df)
    print(f"[4B] zero-shot: {len(base4_res)} примеров за {time.time() - t0:.0f} с")
    experiments.append(evaluate(base4_res, "[Qwen3-4B] zero-shot baseline"))

    t0 = time.time()
    groups4 = run_stage1(eval_df, build_prompt_stage1_v3)
    two4_res = run_stage2(eval_df, groups4, build_prompt_stage2_v3)
    print(f"[4B] two-stage v3: за {time.time() - t0:.0f} с")
    print(f"[4B] этап 1, верная тема: {(groups4 == true_groups).mean():.3f}")
    experiments.append(evaluate(two4_res, "[Qwen3-4B] two-stage v3"))
    right4 = two4_res["pred_group"] == true_groups
    print(f"[4B] этап 2 при верной теме: {(two4_res['pred_label'] == two4_res['label_name'])[right4].mean():.2f}")

    base4_res.to_csv("eval_baseline_4b.csv", index=False)
    two4_res.to_csv("eval_two_stage_v3_4b.csv", index=False)
    print("\nСводная таблица экспериментов (eval_df, n=%d):" % len(eval_df))
    print(pd.DataFrame(experiments).drop_duplicates("experiment", keep="last").to_string(index=False))
else:
    print("RUN_4B = False, ячейка пропущена")
# ---- Конец кода ----

Загружаем Qwen/Qwen3-4B ...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

annotate:   0%|          | 0/200 [00:00<?, ?it/s]

[4B] zero-shot: 200 примеров за 77 с
{'experiment': '[Qwen3-4B] zero-shot baseline', 'n': 200, 'accuracy': np.float64(0.62), 'format_error_rate': np.float64(0.015)}


stage-1:   0%|          | 0/200 [00:00<?, ?it/s]

stage-2:   0%|          | 0/200 [00:00<?, ?it/s]

[4B] two-stage v3: за 154 с
[4B] этап 1, верная тема: 0.800
{'experiment': '[Qwen3-4B] two-stage v3', 'n': 200, 'accuracy': np.float64(0.675), 'format_error_rate': np.float64(0.01)}
[4B] этап 2 при верной теме: 0.84

Сводная таблица экспериментов (eval_df, n=200):
                               experiment   n  accuracy  format_error_rate
                       zero-shot baseline 200     0.490              0.040
           few-shot (1 example per label) 200     0.550              0.050
                            CoT reasoning 200     0.480              0.050
              overlap majority n=3, T=0.7 200     0.485              0.040
      two-stage v2: top-2 groups -> label 200     0.595              0.020
       two-stage v2: top-1 group -> label 200     0.595              0.010
             two-stage v3a: top-1 + rules 200     0.575              0.035
  two-stage v3b: top-1 + rules + codebook 200     0.600              0.020
two-stage v3: stage1 v2 + stage2 codebook 200     0.625     

In [21]:
# резюме по разделу 6

print("""
    Самый лучший результат дало уменьшение вариантов выбора для относительно простой модели.
    примерно с десяток групп и столько же вариантов в каждой группе.
    Варианты с выбором нескольких групп себя не оправдали.
    Итоговая максимальная accuracy 0,67 на 100 самплов с seed 2024 и 0,60 на 200 с seed 42.
    Для модели 4B база 0,62 и two-stage 0,675, время почти не выросло.
""")



    Самый лучший результат дало уменьшение вариантов выбора для относительно простой модели.
    примерно с десяток групп и столько же вариантов в каждой группе.
    Варианты с выбором нескольких групп себя не оправдали.
    Итоговая максимальная accuracy 0,67 на 100 самплов с seed 2024 и 0,60 на 200 с seed 42.
    Для модели 4B база 0,62 и two-stage 0,675, время почти не выросло.



## 7.1. Уверенность ответа. (11 баллов)

Посчитаем уверенность модели ответов по модели. Полезно для гибкой схемы разметки, когда более сложные примеры отправляются на разметку асессорам, либо на модель побольше, или на доп разметку с доп перекрытием.

Рабочий бейзлайн — exp(mean(logprob)) только по токенам итоговой метки. Не включаем reasoning, JSON-ключи и скобки, промпт и EOS. Это score для ранжирования ответов, а не вероятность правильности класса.

Здесь нужно написать функцию для взятия уверенности модели, как указано ниже, и далее показать, что числовая «уверенность» (confidence), посчитанная по лог-вероятностям токенов, действительно коррелирует с тем, ошиблась модель или нет. Ожидается получение AUC>=0.6


https://cookbook.openai.com/examples/using_logprobs

Для reasoning-модели проще начать с отключённого thinking; у Qwen3 это enable_thinking=False в tokenizer.apply_chat_template. Если получаете метку отдельным вызовом, оценивайте именно метку и confidence этого вызова. Покажите 3 примера: ответ → метка → выбранные токены → confidence. Если все ответы правильные или все неправильные, ROC-AUC не определён — укажите это.

Для базового решения достаточно отдельного вызова с ответом только в виде метки — извлекать токены метки из JSON или reasoning не обязательно. Покажите ROC-AUC для всех ответов и отдельно для валидных меток: нулевой confidence у ошибок формата сам по себе может улучшать общий AUC. Цель 0.6 — ориентир; корректный отрицательный результат с объяснением тоже принимается.


In [22]:

# ---- Ваш код здесь ----
# Уверенность = exp(mean(logprob)) только по токенам метки внутри JSON-ответа.
# Берём zero-shot промпт baseline (build_prompt): один вызов, метка в JSON.
# Токены метки находим по символьным границам: склеиваем сгенерированные токены
# в строку, ищем в ней позицию метки после ключа "label" и берём токены, которые
# пересекают этот отрезок. JSON-обвязка, кавычки и EOS в расчёт не попадают.
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score


def _parse_field(raw: str, key: str, allowed: set) -> str | None:
    """Один JSON (можно в блоке ```json), поле key со значением из allowed, иначе None."""
    m = JSON_RE.match(raw)
    if not m:
        return None
    try:
        val = json.loads(m.group(1)).get(key)
    except (json.JSONDecodeError, AttributeError):
        return None
    return val.strip() if isinstance(val, str) and val.strip() in allowed else None


def _label_token_idx(pieces: list[str], label: str) -> list[int]:
    """Индексы токенов, пересекающих отрезок с меткой в склеенном ответе."""
    full = "".join(pieces)
    key = full.find("label")
    start = full.find(label, key + len("label") if key >= 0 else 0)
    if start < 0:
        return []
    end = start + len(label)
    idx, pos = [], 0
    for i, p in enumerate(pieces):
        a, b = pos, pos + len(p)
        pos = b
        if b > start and a < end and p.strip():
            idx.append(i)
    return idx


@torch.inference_mode()
def annotate_conf(text: str,
                  max_new_tokens: int = 32,
                  prompt_fn=build_prompt,
                  allowed: set | None = None,
                  ) -> tuple[str | None, float, int, str, list[str]]:
    """
    Размечает один запрос при помощи LLM и сразу возвращает числовую
    «уверенность» предсказания на основе лог-вероятностей сгенерированных токенов.

    Логика: формируем prompt -> generate(output_scores=True) -> парсим метку из JSON
    (не распарсилась или не из списка: label=None, confidence=0, corrupted=1) ->
    confidence = exp(mean(log p)) только по токенам метки, без JSON-обвязки, промпта и EOS.
    Это score для ранжирования ответов, а не вероятность правильности класса.

    Параметры: text - запрос; max_new_tokens - лимит генерации; prompt_fn - функция
    построения промпта (по умолчанию zero-shot baseline); allowed - допустимые метки
    (по умолчанию все label_names; для второго этапа двухэтапной схемы - метки темы).

    Возвращает: label, confidence (0-1), corrupted (0/1), raw_generation,
    label_tokens - список токенов, по которым посчитан confidence (для отладки).
    """
    allowed = LABEL_SET if allowed is None else allowed
    messages = [{"role": "user", "content": prompt_fn(text)}]
    input_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, enable_thinking=False, return_tensors="pt",
    ).to(device)
    out = model.generate(
        input_ids, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.eos_token_id, output_scores=True, return_dict_in_generate=True,
    )
    gen_ids = out.sequences[0, input_ids.shape[1]:]
    # logprob выбранного токена на каждом шаге генерации
    logprobs = model.compute_transition_scores(out.sequences, out.scores, normalize_logits=True)[0]
    raw = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    label = _parse_field(raw, "label", allowed)
    if label is None:
        return None, 0.0, 1, raw, []
    pieces = [tokenizer.decode([t], skip_special_tokens=True) for t in gen_ids]
    idx = _label_token_idx(pieces, label)
    if not idx:  # метка распарсилась, но токены не нашлись - считаем ошибкой формата
        return None, 0.0, 1, raw, []
    confidence = float(torch.exp(logprobs[idx].float().mean()))
    return label, confidence, 0, raw, [pieces[i] for i in idx]


# Прогон на тех же 100 примерах eval_df.
t0 = time.time()
rows = []
for text in tqdm(eval_df["text"], desc="annotate_conf"):
    label, conf, corrupted, raw, toks = annotate_conf(text)
    rows.append({"pred_label": label, "confidence": conf, "corrupted": corrupted,
                 "raw": raw, "label_tokens": toks})
conf_res = pd.concat([eval_df.reset_index(drop=True), pd.DataFrame(rows)], axis=1)
conf_res["match"] = conf_res["pred_label"] == conf_res["label_name"]
print(f"annotate_conf: {len(conf_res)} примеров за {time.time() - t0:.0f} с")
evaluate(conf_res, "zero-shot + confidence (тот же промпт, что baseline)")
print(f"Совпадение меток с baseline_res: {(conf_res['pred_label'] == baseline_res['pred_label']).mean():.2f} "
      "(greedy, должно быть ~1.0)")

# 3 примера: ответ -> метка -> выбранные токены -> confidence (верный, неверный, ошибка формата если есть).
show_idx = [conf_res[conf_res["match"]].index[0], conf_res[~conf_res["match"] & (conf_res["corrupted"] == 0)].index[0]]
if (conf_res["corrupted"] == 1).any():
    show_idx.append(conf_res[conf_res["corrupted"] == 1].index[0])
for i in show_idx:
    r = conf_res.loc[i]
    print(f"\nТекст: {r['text'][:80]}\n  ответ: {r['raw'][:80]}\n  метка: {r['pred_label']} (истинная: {r['label_name']}, "
          f"верно: {r['match']})\n  токены: {r['label_tokens']}\n  confidence: {r['confidence']:.3f}")

conf_res.to_csv("eval_conf.csv", index=False)
# ---- Конец кода ----

annotate_conf:   0%|          | 0/200 [00:00<?, ?it/s]

annotate_conf: 200 примеров за 79 с
{'experiment': 'zero-shot + confidence (тот же промпт, что baseline)', 'n': 200, 'accuracy': np.float64(0.62), 'format_error_rate': np.float64(0.015)}
Совпадение меток с baseline_res: 0.56 (greedy, должно быть ~1.0)

Текст: How do I link this new card?
  ответ: {"label": "card_linking"}
  метка: card_linking (истинная: card_linking, верно: True)
  токены: ['card', '_link', 'ing']
  confidence: 1.000

Текст: How do I retrieve my card from the machine?
  ответ: {"label": "card_arrival"}
  метка: card_arrival (истинная: card_swallowed, верно: False)
  токены: ['card', '_arr', 'ival']
  confidence: 0.946

Текст: Can you help me with a weird charge?  It's a pound charge that never goes away f
  ответ: {"label": "pending_transaction"}
  метка: None (истинная: extra_charge_on_statement, верно: False)
  токены: []
  confidence: 0.000


In [23]:
# ---- Ваш код здесь ----
# ROC-AUC: умеет ли confidence отделять верные ответы от ошибок.
# y_true_bin = 1, если модель угадала; y_score = confidence.
# Считаем дважды: по всем ответам (ошибки формата имеют confidence=0 и попадают
# в "правильный" конец ранжирования, что само по себе завышает AUC) и только по валидным меткам.


def safe_auc(y, score) -> float:
    """ROC-AUC или NaN с пояснением, если в y один класс (AUC не определён)."""
    if pd.Series(y).nunique() < 2:
        print("  ROC-AUC не определён: все ответы верные или все неверные")
        return float("nan")
    return roc_auc_score(y, score)


y_all = conf_res["match"].astype(int)
auc_all = safe_auc(y_all, conf_res["confidence"])
valid = conf_res[conf_res["corrupted"] == 0]
auc_valid = safe_auc(valid["match"].astype(int), valid["confidence"])
print(f"ROC-AUC confidence: все ответы (n={len(conf_res)}) = {auc_all:.3f}; "
      f"только валидные метки (n={len(valid)}) = {auc_valid:.3f}")
print(f"Средний confidence: верные = {valid.loc[valid['match'], 'confidence'].mean():.3f}, "
      f"ошибки = {valid.loc[~valid['match'], 'confidence'].mean():.3f}")

# Accuracy по корзинам confidence: видно, растёт ли точность с уверенностью.
bins = [0, 0.5, 0.8, 0.9, 0.95, 0.99, 1.0001]
conf_res["conf_bin"] = pd.cut(conf_res["confidence"], bins, include_lowest=True)
print("\nAccuracy по корзинам confidence:")
print(conf_res.groupby("conf_bin", observed=True)["match"].agg(n="size", accuracy="mean").round(3).to_string())

# Что будет, если доверять LLM только выше порога: точность среди доверенных и их доля.
print("\nПорог -> accuracy среди ответов с confidence >= порога, доля таких ответов:")
for thr in [0.5, 0.8, 0.9, 0.95, 0.99]:
    sel = conf_res[conf_res["confidence"] >= thr]
    print(f"  thr={thr:.2f}: accuracy={sel['match'].mean():.3f}, coverage={len(sel) / len(conf_res):.2f}")

# Для сравнения: согласованность прогонов из overlap (раздел 6.3) как альтернативный score.
if "overlap_res" in globals():
    print(f"\nROC-AUC agreement (overlap, n=3): {safe_auc(overlap_res['match'].astype(int), overlap_res['agreement']):.3f}")

conf_res.to_csv("eval_conf.csv", index=False)
# ---- Конец кода ----

ROC-AUC confidence: все ответы (n=200) = 0.751; только валидные метки (n=197) = 0.741
Средний confidence: верные = 0.991, ошибки = 0.972

Accuracy по корзинам confidence:
                 n  accuracy
conf_bin                    
(-0.001, 0.5]    3     0.000
(0.5, 0.8]       2     0.500
(0.8, 0.9]      12     0.417
(0.9, 0.95]      8     0.000
(0.95, 0.99]    14     0.429
(0.99, 1.0]    161     0.696

Порог -> accuracy среди ответов с confidence >= порога, доля таких ответов:
  thr=0.50: accuracy=0.629, coverage=0.98
  thr=0.80: accuracy=0.631, coverage=0.97
  thr=0.90: accuracy=0.645, coverage=0.92
  thr=0.95: accuracy=0.674, coverage=0.88
  thr=0.99: accuracy=0.696, coverage=0.81

ROC-AUC agreement (overlap, n=3): 0.597


In [24]:
# 7.1, второй проход: confidence на втором этапе двухэтапной схемы + разрыв между кандидатами

# ---- Ваш код здесь ----
# Первый проход (conf_res): zero-shot промпт на 77 меток, score = exp(mean logprob) по токенам метки.
# AUC 0.70, но 76% ответов имеют confidence > 0.99 и внутри них ранжирования нет.
# Здесь два изменения, эффект каждого измеряем отдельно:
#  1) промпт второго этапа двухэтапной схемы: список из 4-14 меток темы вместо 77,
#     темы берём из уже посчитанного two_res (этап 1 заново не запускаем);
#  2) новый score - разрыв (margin) между выбранным токеном и вторым кандидатом на шагах
#     генерации метки: p(top1) - p(top2). Берём margin на первом токене метки (там решается
#     выбор между семействами меток) и минимальный по всем токенам метки (самое неуверенное
#     место внутри метки, например card_|arrival против card_|delivery).
# Оба промпта прогоняем через одну функцию, чтобы разделить эффект промпта и эффект score.


def _label_scores(scores, gen_ids, idx) -> tuple[float, float, float]:
    """По логитам шагов генерации и индексам токенов метки: (exp(mean logprob), margin_first, margin_min)."""
    lp = torch.stack([F.log_softmax(scores[i][0].float(), dim=-1) for i in idx])  # [n_tokens, vocab]
    chosen = lp.gather(1, gen_ids[idx].view(-1, 1)).squeeze(1)                    # logprob выбранных токенов
    top2 = lp.topk(2, dim=-1).values.exp()                                         # p(top1), p(top2) на каждом шаге
    margins = top2[:, 0] - top2[:, 1]
    return float(chosen.mean().exp()), float(margins[0]), float(margins.min())


@torch.inference_mode()
def annotate_conf2(text: str, prompt_fn=build_prompt, allowed: set | None = None,
                   max_new_tokens: int = 32) -> dict:
    """Как annotate_conf, но возвращает три score сразу: conf_logprob, margin_first, margin_min."""
    allowed = LABEL_SET if allowed is None else allowed
    messages = [{"role": "user", "content": prompt_fn(text)}]
    input_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, enable_thinking=False, return_tensors="pt",
    ).to(device)
    out = model.generate(
        input_ids, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.eos_token_id, output_scores=True, return_dict_in_generate=True,
    )
    gen_ids = out.sequences[0, input_ids.shape[1]:]
    raw = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    bad = {"pred_label": None, "corrupted": 1, "conf_logprob": 0.0, "margin_first": 0.0, "margin_min": 0.0, "raw": raw}
    label = _parse_field(raw, "label", allowed)
    if label is None:
        return bad
    pieces = [tokenizer.decode([t], skip_special_tokens=True) for t in gen_ids]
    idx = _label_token_idx(pieces, label)
    if not idx:
        return bad
    conf, m_first, m_min = _label_scores(out.scores, gen_ids, idx)
    return {"pred_label": label, "corrupted": 0, "conf_logprob": conf,
            "margin_first": m_first, "margin_min": m_min, "raw": raw}


def run_conf2(name: str, prompt_for_row) -> pd.DataFrame:
    """prompt_for_row(i, text) -> (prompt_fn, allowed) для i-й строки eval_df."""
    rows = []
    for i, text in enumerate(tqdm(eval_df["text"], desc=name)):
        pf, allowed = prompt_for_row(i, text)
        rows.append(annotate_conf2(text, pf, allowed))
    res = pd.concat([eval_df.reset_index(drop=True), pd.DataFrame(rows)], axis=1)
    res["match"] = res["pred_label"] == res["label_name"]
    return res


# Темы для второго этапа: top-1 из two_res (top-1 и top-2 дали одинаковые 0.67, список у top-1 короче).
# Для top-2 замените на two_res["pred_groups"].
CONF_GROUPS = two_res["group_top1"]


def stage2_for_row(i, text):
    g = CONF_GROUPS.iloc[i]
    groups = [g] if isinstance(g, str) else list(g or [])
    if not groups:  # этап 1 не дал темы (в two_res таких нет) - запасной вариант zero-shot
        return build_prompt, LABEL_SET
    return (lambda t: build_prompt_stage2(t, groups)), {l for gg in groups for l in GROUPS[gg][1]}


t0 = time.time()
conf_zero = run_conf2("zero-shot", lambda i, t: (build_prompt, LABEL_SET))
conf_stage2 = run_conf2("stage-2", stage2_for_row)
print(f"Два прогона по {len(eval_df)} примеров за {time.time() - t0:.0f} с")
evaluate(conf_stage2, "two-stage top-1 + confidence (stage 2 re-run)")


def score_table(res: pd.DataFrame, pass_name: str, scores: list[str]) -> list[dict]:
    """AUC (все / валидные) и accuracy среди самых уверенных при фиксированном покрытии.
    Покрытие по рангу score, поэтому строки с разными score сравнимы между собой."""
    y = res["match"].astype(int)
    valid = res[res["corrupted"] == 0]
    out = []
    for s in scores:
        row = {"pass": pass_name, "score": s, "accuracy": res["match"].mean(),
               "auc_all": safe_auc(y, res[s]), "auc_valid": safe_auc(valid["match"].astype(int), valid[s])}
        for cov in (0.5, 0.75, 0.9):
            top = res.sort_values(s, ascending=False).head(int(round(cov * len(res))))
            row[f"acc@cov{cov}"] = top["match"].mean()
        out.append(row)
    return out


SCORES = ["conf_logprob", "margin_first", "margin_min"]
rows = score_table(conf_res.assign(conf_logprob=conf_res["confidence"]), "1st pass: zero-shot", ["conf_logprob"])
rows += score_table(conf_zero, "zero-shot", SCORES)
rows += score_table(conf_stage2, "stage-2 (two-stage top-1)", SCORES)
conf_table = pd.DataFrame(rows).round(3)
print("\nСравнение score (acc@covX = accuracy среди X самых уверенных ответов по этому score):")
print(conf_table.to_string(index=False))

# Насколько вырожден каждый score: квантили. Чем шире разброс, тем полезнее score для порога.
print("\nРаспределение score, stage-2:")
print(conf_stage2[SCORES].describe(percentiles=[0.1, 0.25, 0.5, 0.75]).round(3).T.to_string())
print("\nРаспределение score, zero-shot:")
print(conf_zero[SCORES].describe(percentiles=[0.1, 0.25, 0.5, 0.75]).round(3).T.to_string())

conf_zero.to_csv("eval_conf_zero.csv", index=False)
conf_stage2.to_csv("eval_conf_stage2.csv", index=False)
# ---- Конец кода ----

zero-shot:   0%|          | 0/200 [00:00<?, ?it/s]

stage-2:   0%|          | 0/200 [00:00<?, ?it/s]

Два прогона по 200 примеров за 156 с
{'experiment': 'two-stage top-1 + confidence (stage 2 re-run)', 'n': 200, 'accuracy': np.float64(0.675), 'format_error_rate': np.float64(0.015)}

Сравнение score (acc@covX = accuracy среди X самых уверенных ответов по этому score):
                     pass        score  accuracy  auc_all  auc_valid  acc@cov0.5  acc@cov0.75  acc@cov0.9
      1st pass: zero-shot conf_logprob     0.620    0.751      0.741        0.82        0.713       0.656
                zero-shot conf_logprob     0.620    0.751      0.741        0.82        0.713       0.656
                zero-shot margin_first     0.620    0.763      0.753        0.80        0.727       0.667
                zero-shot   margin_min     0.620    0.747      0.737        0.83        0.713       0.656
stage-2 (two-stage top-1) conf_logprob     0.675    0.686      0.671        0.78        0.787       0.711
stage-2 (two-stage top-1) margin_first     0.675    0.767      0.756        0.82        0.813  

## Вывод по разделу 7

Это результат первого цикла, 100 самплов eval_df = test_df.sample(n=100, random_state=2024)

контроль совпал, промпт второго этапа улучшил и метку, и score, margin равноценен логпробу на коротком списке и хуже на длинном

```code
Два прогона по 100 примеров за 57 с
{'experiment': 'two-stage top-1 + confidence (stage 2 re-run)', 'n': 100, 'accuracy': np.float64(0.67), 'format_error_rate': np.float64(0.04)}

Сравнение score (acc@covX = accuracy среди X самых уверенных ответов по этому score):
                     pass        score  accuracy  auc_all  auc_valid  acc@cov0.5  acc@cov0.75  acc@cov0.9
      1st pass: zero-shot conf_logprob      0.59    0.700      0.685        0.74        0.667       0.656
                zero-shot conf_logprob      0.59    0.701      0.686        0.74        0.667       0.656
                zero-shot margin_first      0.59    0.666      0.649        0.72        0.653       0.633
                zero-shot   margin_min      0.59    0.687      0.671        0.72        0.667       0.656
stage-2 (two-stage top-1) conf_logprob      0.67    0.737      0.701        0.84        0.787       0.733
stage-2 (two-stage top-1) margin_first      0.67    0.759      0.725        0.84        0.800       0.733
stage-2 (two-stage top-1)   margin_min      0.67    0.720      0.681        0.84        0.787       0.722

Распределение score, stage-2:
              count   mean    std  min    10%    25%  50%  75%  max
conf_logprob  100.0  0.949  0.198  0.0  0.933  0.999  1.0  1.0  1.0
margin_first  100.0  0.929  0.231  0.0  0.938  1.000  1.0  1.0  1.0
margin_min    100.0  0.900  0.254  0.0  0.608  0.993  1.0  1.0  1.0

Распределение score, zero-shot:
              count   mean    std  min    10%    25%  50%  75%  max
conf_logprob  100.0  0.961  0.145  0.0  0.895  0.993  1.0  1.0  1.0
margin_first  100.0  0.917  0.212  0.0  0.758  0.980  1.0  1.0  1.0
margin_min    100.0  0.884  0.242  0.0  0.434  0.957  1.0  1.0  1.0
```

Теперь проверю сходимость результата: 200 самплов с сидом 42

```code
Два прогона по 200 примеров за 118 с
{'experiment': 'two-stage top-1 + confidence (stage 2 re-run)', 'n': 200, 'accuracy': np.float64(0.595), 'format_error_rate': np.float64(0.01)}

Сравнение score (acc@covX = accuracy среди X самых уверенных ответов по этому score):
                     pass        score  accuracy  auc_all  auc_valid  acc@cov0.5  acc@cov0.75  acc@cov0.9
      1st pass: zero-shot conf_logprob     0.490    0.781      0.762        0.72        0.587       0.533
                zero-shot conf_logprob     0.490    0.780      0.762        0.72        0.587       0.533
                zero-shot margin_first     0.490    0.716      0.691        0.67        0.567       0.528
                zero-shot   margin_min     0.490    0.779      0.760        0.72        0.593       0.533
stage-2 (two-stage top-1) conf_logprob     0.595    0.721      0.714        0.75        0.713       0.617
stage-2 (two-stage top-1) margin_first     0.595    0.732      0.726        0.76        0.713       0.633
stage-2 (two-stage top-1)   margin_min     0.595    0.710      0.703        0.75        0.707       0.617

Распределение score, stage-2:
              count   mean    std  min    10%    25%  50%  75%  max
conf_logprob  200.0  0.978  0.104  0.0  0.950  0.999  1.0  1.0  1.0
margin_first  200.0  0.940  0.182  0.0  0.800  1.000  1.0  1.0  1.0
margin_min    200.0  0.921  0.203  0.0  0.678  0.996  1.0  1.0  1.0

Распределение score, zero-shot:
              count   mean    std  min    10%    25%  50%  75%  max
conf_logprob  200.0  0.938  0.200  0.0  0.870  0.993  1.0  1.0  1.0
margin_first  200.0  0.893  0.264  0.0  0.483  0.994  1.0  1.0  1.0
margin_min    200.0  0.849  0.299  0.0  0.318  0.954  1.0  1.0  1.0
```

AUC 0.78 достигнут и устойчив на двух выборках; confidence отделяет четверть сомнительных ответов, где точность около 0.2, от остальных, где около 0.6; внутри уверенных ответов шкалы нет; для гибрида с человеком уровня 0.8 этого недостаточно

Модель Qwen3-4B дает лучшие результаты: 

```code
Два прогона по 200 примеров за 156 с
{'experiment': 'two-stage top-1 + confidence (stage 2 re-run)', 'n': 200, 'accuracy': np.float64(0.675), 'format_error_rate': np.float64(0.015)}

Сравнение score (acc@covX = accuracy среди X самых уверенных ответов по этому score):
                     pass        score  accuracy  auc_all  auc_valid  acc@cov0.5  acc@cov0.75  acc@cov0.9
      1st pass: zero-shot conf_logprob     0.620    0.751      0.741        0.82        0.713       0.656
                zero-shot conf_logprob     0.620    0.751      0.741        0.82        0.713       0.656
                zero-shot margin_first     0.620    0.763      0.753        0.80        0.727       0.667
                zero-shot   margin_min     0.620    0.747      0.737        0.83        0.713       0.656
stage-2 (two-stage top-1) conf_logprob     0.675    0.686      0.671        0.78        0.787       0.711
stage-2 (two-stage top-1) margin_first     0.675    0.767      0.756        0.82        0.813       0.717
stage-2 (two-stage top-1)   margin_min     0.675    0.689      0.674        0.80        0.780       0.711

Распределение score, stage-2:
              count   mean    std  min    10%  25%  50%  75%  max
conf_logprob  200.0  0.972  0.127  0.0  0.945  1.0  1.0  1.0  1.0
margin_first  200.0  0.930  0.215  0.0  0.795  1.0  1.0  1.0  1.0
margin_min    200.0  0.918  0.230  0.0  0.678  1.0  1.0  1.0  1.0

Распределение score, zero-shot:
              count   mean    std  min    10%    25%  50%  75%  max
conf_logprob  200.0  0.969  0.127  0.0  0.939  0.998  1.0  1.0  1.0
margin_first  200.0  0.923  0.221  0.0  0.795  0.995  1.0  1.0  1.0
margin_min    200.0  0.898  0.242  0.0  0.520  0.980  1.0  1.0  1.0
```

## 8.1. Human-in-the-loop с шумным разметчиком

В реальных задачах разметку часто делают не идеальные эксперты, а обычные асессоры — они тоже ошибаются. Для достижения хорошего качества с ними используется разметка с перкрытием по N accесорам, а затем агрегируется разметка (например majority vote). Таким образом получаем **агрегированную метку** и меру согласованности ассесоров - насколько они сходятся в решении (например 0.75 - из 4 ассессоров 3 выбрали итоговую метку).

Согласованность не гарантирует правильность: чтобы использовать разметку как gold, нужна дополнительная проверка качества.

Часть разметки можно переложить на LLM, если уметь оценивать его уверенность.

В этом задании мы смоделируем простую схему:

- есть **gold-метка** `true_label` (используем только для оценки качества),
- есть один **"человек"-разметчик** с ошибками (`human_label`),
- есть предсказания LLM: `pred_label` и `confidence`.

Мы хотим построить **гибридную систему**:

- по умолчанию используем метку человека (`human_label`);
- если `confidence >= threshold` — считаем, что LLM очень уверен и берём его метку (`pred_label`).

#### 8.2.1. Симуляция "шумного" разметчика (3 балла)

Реализуйте функцию, которая по gold-меткам `true_labels` возвращает `human_labels`:

- с вероятностью `1 - error_rate` берётся правильная метка,
- с вероятностью `error_rate` — случайная *другая* метка из `label_names`  
  (можно взять, например, `error_rate = 0.20`).

1. Реализуйте функцию `simulate_noisy_human(true_labels, label_names, error_rate=0.20, random_state=42)`.
2. Посчитайте accuracy такого разметчика относительно `true_labels`.

Замечание: в реальных задачах расхождения между разметчиками и ошибочно проставленные метки зачастую не случайны. В данном задании симулируем шум для упрощения.
Сгенерируйте human_labels один раз и используйте их при всех порогах. Это вероятность ошибки 20%, а не требование получить ровно 20% ошибочных строк.

Здесь 20% — выбранный сценарий ошибки одного разметчика, а не измеренное качество людей на Banking77 и не процент несогласия между ними. В исходном примере COLING 2025 шум зависел от согласованности разметки каждого объекта; здесь используем одну вероятность для простоты.


In [ ]:
# ---- Ваш код здесь ----

# ---- Конец кода ----


#### 8.2.2. Гибридная схема разметки (4 балла)

Реализуйте функцию, которая комбинирует разметку человека и LLM по порогу уверенности:

- по умолчанию использует `human_labels`,
- если `pred_labels[i]` — допустимая метка и `confidence[i] >= threshold` — вместо метки человека берёт `pred_labels[i]`; невалидный ответ всегда остаётся человеку,
- возвращает:
  - `overall_acc` — итоговую accuracy гибридной разметки (относительно `true_labels`),
  - `coverage` — долю объектов, где использовали LLM (от 0 до 1).

1. Реализуйте функцию  
   `simulate_hybrid(pred_labels, human_labels, true_labels, confidence, threshold)`.
2. Проверьте её на небольшом игрушечном примере

In [ ]:
# ---- Ваш код здесь ----

# ---- Конец кода ----


#### 8.2.3. Подбор порога и анализ trade-off (3 балла)

1. Для `threshold` из диапазона `[0.0, 1.0]` с шагом 0.01:
   - посчитайте `overall_acc` и `coverage`;
   - выведите таблицу с колонками `threshold`, `accuracy`, `coverage`  
     (по желанию можно дополнительно построить график `accuracy` vs `coverage`).
2. Найдите значение `threshold`, при котором:
   - качество гибридной разметки **не ниже** заданного уровня (например, accuracy ≥ 0.8 *относительно gold*),
   - а `coverage` максимально возможен.
Если допустимого порога нет, явно сообщите об этом; это тоже корректный результат. Подбор на этой же выборке — учебное описательное сравнение, а не независимая оценка выбранного порога.


In [ ]:
# ---- Ваш код здесь ----

# ---- Конец кода ----


#### 8.2.4. Сравнение схем разметки (2 балла)

1. Посчитайте и сравните:
   - accuracy только человека (`human_labels`),
   - accuracy только LLM (`pred_labels`),
   - лучшую точку гибрида.
2. Кратко (2–3 предложения) ответьте:
   - появилось ли преимущество гибрида: как изменились accuracy и доля запросов человеку относительно «чисто человек» и «чисто LLM»?
   - в каких кейсах такая схема human-in-the-loop может быть особенно полезна?

Перед сдачей сохраните ноутбук с outputs. В конце покажите таблицу экспериментов, accuracy/AUC confidence и таблицу выбора порога; укажите модель, размер выборки и seed.


In [ ]:
# ---- Ваш код здесь ----

# ---- Конец кода ----


## Итоги домашки

В этой работе мы посмотрели на разметку как на систему, где есть и люди, и LLM.

Главное, что нужно вынести:
- LLM можно использовать как разметчика (при этом важно следить за качеством ), можно улучшать промт за счет различных прдеставленных способов.
- Оценку **уверенности** по logprobs нужно проверить на данных, прежде чем решать, где доверять модели, а где подключать человека.
- Гибридную схему human-in-the-loop сравниваем с «только крауд» и «только LLM» по качеству и доле автоматизации: выигрыш не гарантирован.
- Эти идеи масштабируются дальше: улучшение промптов, дообучение модели, active learning и более умные пайплайны разметки.


```Code
Разметчик из LLM можно получить, но простые модели дают не очень хороший результат. 
Confidency на 1,7B максимум 0,635, на 4B 0,675.
схема	        1.7B	4B
zero-shot	    0.49	0.62
two-stage v3	0.635	0.675

AUC на модели 4B 0.75 при цели 0.6.

С человеком не сделал, не осознал задание.

```